# build_vocab: the PolyPlex v2 text vocabulary

This notebook decides the final text vocabulary (SPEC S2.3, S12.1a, S12.4, S12.6). Running it top to bottom produces the
same vocabulary again. Every step records in a table what it added, by which rule and from which data.

**Order of inclusion (the three tiers of S2.3)**
1. **Character floor**: every standard character of each language becomes a single token, with no measurement. The check is
   0% byte-fallback tokens on spoken data.
2. **Frequent word forms**: short, frequent spoken forms. The amount is set by the word cut rate (S2.2).
3. **Subwords**: pieces that cut the words tier 2 lacks into fewer tokens; sized by the cut rate where tier 2 does not
   reach the target, small elsewhere.

**Blocks (S12.1a, S12.4)**
- Base block: all 32,000 Helium tokens, plus Qwen3's digits, punctuation, symbols and byte fragments. Always computed.
- Script blocks: Qwen3 tokens with no identical string in Helium, grouped by script.
- Language blocks: the tokens the three tiers add.

**Output**: one result file per step under `.cache/huggingface/runs/vocab/`.

The code stays in this notebook (AGENTS.md 6.1). Run it from the repository root.

In [1]:
import collections
import json
import os
import re
import unicodedata
from dataclasses import dataclass
import urllib.request
from pathlib import Path

os.environ.setdefault("HF_HOME", str(Path(".cache/huggingface").resolve()))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from IPython.display import Markdown, display
from transformers import AutoTokenizer

from polyplex_research.utils.assemble import load_base_tokenizer

HELIUM = "nvidia/personaplex-7b-v1"
QWEN = "Qwen/Qwen3-8B"
# CLDR release the standard characters are read from. A fixed tag keeps the result reproducible.
CLDR_RELEASE = "48.2.3"
OUT = Path(".cache/huggingface/runs/vocab")


@dataclass(frozen=True)
class Qwen3Language:
    """One language or dialect Qwen3 lists, with its code in each corpus that has it (None when the corpus lacks it)."""

    name: str
    fineweb2: str | None
    wikipedia: str | None
    fleurs: str | None


# The 119 languages and dialects of the Qwen3 release (https://qwenlm.github.io/blog/qwen3/), in its order and grouped by
# its language families. Codes checked against the Hugging Face listings of FineWeb-2, Wikipedia (20231101) and FLEURS.
# A variety without a corpus of its own shares its parent's code, as Dari shares Persian's, or has none.
QWEN3_LANGUAGES = (
    # Indo-European
    Qwen3Language("English", None, "en", "en_us"),
    Qwen3Language("French", "fra_Latn", "fr", "fr_fr"),
    Qwen3Language("Portuguese", "por_Latn", "pt", "pt_br"),
    Qwen3Language("German", "deu_Latn", "de", "de_de"),
    Qwen3Language("Romanian", "ron_Latn", "ro", "ro_ro"),
    Qwen3Language("Swedish", "swe_Latn", "sv", "sv_se"),
    Qwen3Language("Danish", "dan_Latn", "da", "da_dk"),
    Qwen3Language("Bulgarian", "bul_Cyrl", "bg", "bg_bg"),
    Qwen3Language("Russian", "rus_Cyrl", "ru", "ru_ru"),
    Qwen3Language("Czech", "ces_Latn", "cs", "cs_cz"),
    Qwen3Language("Greek", "ell_Grek", "el", "el_gr"),
    Qwen3Language("Ukrainian", "ukr_Cyrl", "uk", "uk_ua"),
    Qwen3Language("Spanish", "spa_Latn", "es", "es_419"),
    Qwen3Language("Dutch", "nld_Latn", "nl", "nl_nl"),
    Qwen3Language("Slovak", "slk_Latn", "sk", "sk_sk"),
    Qwen3Language("Croatian", "hrv_Latn", "hr", "hr_hr"),
    Qwen3Language("Polish", "pol_Latn", "pl", "pl_pl"),
    Qwen3Language("Lithuanian", "lit_Latn", "lt", "lt_lt"),
    Qwen3Language("Norwegian Bokmal", "nob_Latn", "no", "nb_no"),
    Qwen3Language("Norwegian Nynorsk", "nno_Latn", "nn", None),
    Qwen3Language("Persian", "fas_Arab", "fa", "fa_ir"),
    Qwen3Language("Slovenian", "slv_Latn", "sl", "sl_si"),
    Qwen3Language("Gujarati", "guj_Gujr", "gu", "gu_in"),
    Qwen3Language("Latvian", "lvs_Latn", "lv", "lv_lv"),
    Qwen3Language("Italian", "ita_Latn", "it", "it_it"),
    Qwen3Language("Occitan", "oci_Latn", "oc", "oc_fr"),
    Qwen3Language("Nepali", "npi_Deva", "ne", "ne_np"),
    Qwen3Language("Marathi", "mar_Deva", "mr", "mr_in"),
    Qwen3Language("Belarusian", "bel_Cyrl", "be", "be_by"),
    Qwen3Language("Serbian", "srp_Cyrl", "sr", "sr_rs"),
    Qwen3Language("Luxembourgish", "ltz_Latn", "lb", "lb_lu"),
    Qwen3Language("Venetian", "vec_Latn", "vec", None),
    Qwen3Language("Assamese", "asm_Beng", "as", "as_in"),
    Qwen3Language("Welsh", "cym_Latn", "cy", "cy_gb"),
    Qwen3Language("Silesian", "szl_Latn", "szl", None),
    Qwen3Language("Asturian", "ast_Latn", "ast", "ast_es"),
    Qwen3Language("Chhattisgarhi", "hne_Deva", None, None),
    Qwen3Language("Awadhi", "awa_Deva", None, None),
    Qwen3Language("Maithili", "mai_Deva", "mai", None),
    Qwen3Language("Bhojpuri", "bho_Deva", "bh", None),
    Qwen3Language("Sindhi", "snd_Arab", "sd", "sd_in"),
    Qwen3Language("Irish", "gle_Latn", "ga", "ga_ie"),
    Qwen3Language("Faroese", "fao_Latn", "fo", None),
    Qwen3Language("Hindi", "hin_Deva", "hi", "hi_in"),
    Qwen3Language("Punjabi", "pan_Guru", "pa", "pa_in"),
    Qwen3Language("Bengali", "ben_Beng", "bn", "bn_in"),
    Qwen3Language("Oriya", "ory_Orya", "or", "or_in"),
    Qwen3Language("Tajik", "tgk_Cyrl", "tg", "tg_tj"),
    Qwen3Language("Eastern Yiddish", "ydd_Hebr", "yi", None),
    Qwen3Language("Lombard", "lmo_Latn", "lmo", None),
    Qwen3Language("Ligurian", "lij_Latn", "lij", None),
    Qwen3Language("Sicilian", "scn_Latn", "scn", None),
    Qwen3Language("Friulian", "fur_Latn", "fur", None),
    Qwen3Language("Sardinian", "srd_Latn", "sc", None),
    Qwen3Language("Galician", "glg_Latn", "gl", "gl_es"),
    Qwen3Language("Catalan", "cat_Latn", "ca", "ca_es"),
    Qwen3Language("Icelandic", "isl_Latn", "is", "is_is"),
    Qwen3Language("Tosk Albanian", "als_Latn", "sq", None),
    Qwen3Language("Limburgish", "lim_Latn", "li", None),
    Qwen3Language("Dari", "fas_Arab", None, None),
    Qwen3Language("Afrikaans", "afr_Latn", "af", "af_za"),
    Qwen3Language("Macedonian", "mkd_Cyrl", "mk", "mk_mk"),
    Qwen3Language("Sinhala", "sin_Sinh", "si", None),
    Qwen3Language("Urdu", "urd_Arab", "ur", "ur_pk"),
    Qwen3Language("Magahi", "mag_Deva", None, None),
    Qwen3Language("Bosnian", "bos_Latn", "bs", "bs_ba"),
    Qwen3Language("Armenian", "hye_Armn", "hy", "hy_am"),
    # Sino-Tibetan
    Qwen3Language("Simplified Chinese", "cmn_Hani", "zh", "cmn_hans_cn"),
    Qwen3Language("Traditional Chinese", "cmn_Hani", "zh", None),
    Qwen3Language("Cantonese", "yue_Hani", "zh-yue", "yue_hant_hk"),
    Qwen3Language("Burmese", "mya_Mymr", "my", "my_mm"),
    # Afro-Asiatic
    Qwen3Language("Standard Arabic", "arb_Arab", "ar", "ar_eg"),
    Qwen3Language("Najdi Arabic", "ars_Arab", None, None),
    Qwen3Language("Levantine Arabic", "apc_Arab", None, None),
    Qwen3Language("Egyptian Arabic", "arz_Arab", "arz", None),
    Qwen3Language("Moroccan Arabic", "ary_Arab", "ary", None),
    Qwen3Language("Mesopotamian Arabic", "acm_Arab", None, None),
    Qwen3Language("Taizzi-Adeni Arabic", None, None, None),
    Qwen3Language("Tunisian Arabic", "aeb_Arab", None, None),
    Qwen3Language("Hebrew", "heb_Hebr", "he", "he_il"),
    Qwen3Language("Maltese", "mlt_Latn", "mt", "mt_mt"),
    # Austronesian
    Qwen3Language("Indonesian", "ind_Latn", "id", "id_id"),
    Qwen3Language("Malay", "zsm_Latn", "ms", "ms_my"),
    Qwen3Language("Tagalog", "fil_Latn", "tl", "fil_ph"),
    Qwen3Language("Cebuano", "ceb_Latn", "ceb", "ceb_ph"),
    Qwen3Language("Javanese", "jav_Latn", "jv", "jv_id"),
    Qwen3Language("Sundanese", "sun_Latn", "su", None),
    Qwen3Language("Minangkabau", "min_Latn", "min", None),
    Qwen3Language("Balinese", "ban_Latn", "ban", None),
    Qwen3Language("Banjar", "bjn_Latn", "bjn", None),
    Qwen3Language("Pangasinan", "pag_Latn", "pag", None),
    Qwen3Language("Iloko", "ilo_Latn", "ilo", None),
    Qwen3Language("Waray", "war_Latn", "war", None),
    # Dravidian
    Qwen3Language("Tamil", "tam_Taml", "ta", "ta_in"),
    Qwen3Language("Telugu", "tel_Telu", "te", "te_in"),
    Qwen3Language("Kannada", "kan_Knda", "kn", "kn_in"),
    Qwen3Language("Malayalam", "mal_Mlym", "ml", "ml_in"),
    # Turkic
    Qwen3Language("Turkish", "tur_Latn", "tr", "tr_tr"),
    Qwen3Language("North Azerbaijani", "azj_Latn", "az", "az_az"),
    Qwen3Language("Northern Uzbek", "uzn_Latn", "uz", "uz_uz"),
    Qwen3Language("Kazakh", "kaz_Cyrl", "kk", "kk_kz"),
    Qwen3Language("Bashkir", "bak_Cyrl", "ba", None),
    Qwen3Language("Tatar", "tat_Cyrl", "tt", None),
    # Tai-Kadai
    Qwen3Language("Thai", "tha_Thai", "th", "th_th"),
    Qwen3Language("Lao", "lao_Laoo", "lo", "lo_la"),
    # Uralic
    Qwen3Language("Finnish", "fin_Latn", "fi", "fi_fi"),
    Qwen3Language("Estonian", "ekk_Latn", "et", "et_ee"),
    Qwen3Language("Hungarian", "hun_Latn", "hu", "hu_hu"),
    # Austroasiatic
    Qwen3Language("Vietnamese", "vie_Latn", "vi", "vi_vn"),
    Qwen3Language("Khmer", "khm_Khmr", "km", "km_kh"),
    # Other
    Qwen3Language("Japanese", "jpn_Jpan", "ja", "ja_jp"),
    Qwen3Language("Korean", "kor_Hang", "ko", "ko_kr"),
    Qwen3Language("Georgian", "kat_Geor", "ka", "ka_ge"),
    Qwen3Language("Basque", "eus_Latn", "eu", None),
    Qwen3Language("Haitian", "hat_Latn", "ht", None),
    Qwen3Language("Papiamento", "pap_Latn", "pap", None),
    Qwen3Language("Kabuverdianu", "kea_Latn", None, "kea_cv"),
    Qwen3Language("Tok Pisin", "tpi_Latn", "tpi", None),
    Qwen3Language("Swahili", "swh_Latn", "sw", "sw_ke"),
)

OUT.mkdir(parents=True, exist_ok=True)


import time

THREADS = os.cpu_count()
started = time.time()
# Languages are searched independently, each in a forked process with a share of the cores.
WORKERS = max(1, min(32, THREADS // 2))
WORKER_THREADS = max(1, THREADS // WORKERS)


def in_workers(function, items):
    if WORKERS == 1:
        return [function(item) for item in items]
    import multiprocessing

    with multiprocessing.get_context("fork").Pool(WORKERS) as pool:
        return pool.map(function, items, chunksize=1)


def elapsed(label):
    # Also appended to a log file, so a running notebook shows its progress.
    line = f"[{label}] {time.time() - started:,.0f} s since the start"
    print(line, flush=True)
    with open(OUT / "build_vocab_progress.log", "a") as log:
        log.write(f"{time.strftime('%H:%M:%S')} {line}\n")


def table(rows, header):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

W1024 18:59:12.231000 3827323 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1024 18:59:12.301000 3827323 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 1. Matching the base vocabularies

Qwen3 and Helium tokens are matched by surface string. Helium's `▁` (SentencePiece) and Qwen3's `Ġ` (byte BPE) both mean a
leading space. Tokens with the same string are treated as one token and kept as the Helium token (base block, Helium head rows
preserved). Qwen3 tokens missing from Helium are grouped by script into script blocks. Digits, punctuation, symbols and byte
fragments go to the base block.

In [2]:
helium = load_base_tokenizer(HELIUM)
qwen = AutoTokenizer.from_pretrained(QWEN)

helium_surface = {}
helium_special = []
for index in range(len(helium)):
    piece = helium.convert_ids_to_tokens(index)
    if piece.startswith("<") and piece.endswith(">"):
        helium_special.append(piece)
        continue
    helium_surface.setdefault(piece.replace("▁", " "), index)

# Qwen3's added tokens are markers, not text. Its tool-call and thinking markers enter as control tokens; the fill-in-the-middle
# and repository markers are left out (S12.1a). A Qwen3 string equal to a Helium control token (`<s>`) is left out too.
QWEN_CONTROL = ["<tool_call>", "</tool_call>", "<tool_response>", "</tool_response>", "<think>", "</think>"]
qwen_surface = {}
for token, index in qwen.get_vocab().items():
    if index in qwen.all_special_ids or token in qwen.get_added_vocab():
        continue
    surface = qwen.convert_tokens_to_string([token])
    if surface not in helium_special:
        qwen_surface.setdefault(surface, index)

shared = set(helium_surface) & set(qwen_surface)
helium_only = set(helium_surface) - set(qwen_surface)

SCRIPTS = ("LATIN", "CYRILLIC", "ARABIC", "HEBREW", "GREEK", "THAI", "LAO", "KHMER", "MYANMAR", "GEORGIAN", "ARMENIAN",
           "DEVANAGARI", "BENGALI", "GURMUKHI", "GUJARATI", "ORIYA", "TAMIL", "TELUGU", "KANNADA", "MALAYALAM", "SINHALA")


def script_of(ch):
    code = ord(ch)
    if 0xAC00 <= code <= 0xD7A3 or 0x1100 <= code <= 0x11FF or 0x3130 <= code <= 0x318F:
        return "hangul"
    if 0x3040 <= code <= 0x30FF or 0x31F0 <= code <= 0x31FF:
        return "kana"
    if 0x4E00 <= code <= 0x9FFF or 0x3400 <= code <= 0x4DBF or 0xF900 <= code <= 0xFAFF or 0x20000 <= code <= 0x2FFFF:
        return "han"
    if ch.isascii():
        return "latin" if ch.isalpha() else None
    name = unicodedata.name(ch, "")
    for script in SCRIPTS:
        if name.startswith(script):
            return script.lower()
    if unicodedata.category(ch)[0] in "PSZNC":
        return None
    return "other"


def block_of(surface):
    if "�" in surface or not surface.strip():
        return "base"
    scripts = {script_of(ch) for ch in surface} - {None}
    if not scripts:
        return "base"
    if scripts <= {"han", "kana"}:
        return "kana" if "kana" in scripts else "han"
    if len(scripts) == 1:
        return next(iter(scripts))
    return "mixed"


qwen_blocks = collections.defaultdict(list)
for surface, index in qwen_surface.items():
    if surface in shared:
        continue
    qwen_blocks[block_of(surface)].append(surface)

base_size = len(helium) + len(QWEN_CONTROL) + len(qwen_blocks["base"])
table(
    [
        ["Helium vocabulary (with special tokens)", len(helium)],
        ["  of which special tokens", len(helium_special)],
        ["Qwen3 vocabulary (without special and added tokens)", len(qwen_surface)],
        ["Qwen3 tool-call and thinking markers (control tokens)", len(QWEN_CONTROL)],
        ["Tokens with the same string in both (kept as Helium tokens)", len(shared)],
        ["Helium-only tokens", len(helium_only)],
        ["Qwen3-only digits, punctuation, symbols and byte fragments (base block)", len(qwen_blocks["base"])],
        ["**Base block total**", base_size],
    ],
    ["Item", "Count"],
)
table(
    sorted(([name, len(tokens)] for name, tokens in qwen_blocks.items() if name != "base"), key=lambda row: -row[1]),
    ["Script block (Qwen3-only tokens)", "Count"],
)
json.dump({name: sorted(tokens) for name, tokens in qwen_blocks.items()}, open(OUT / "qwen_blocks.json", "w"), ensure_ascii=False)

| Item | Count |
|---|---|
| Helium vocabulary (with special tokens) | 32000 |
|   of which special tokens | 261 |
| Qwen3 vocabulary (without special and added tokens) | 150303 |
| Qwen3 tool-call and thinking markers (control tokens) | 6 |
| Tokens with the same string in both (kept as Helium tokens) | 26471 |
| Helium-only tokens | 5268 |
| Qwen3-only digits, punctuation, symbols and byte fragments (base block) | 8297 |
| **Base block total** | 40303 |

| Script block (Qwen3-only tokens) | Count |
|---|---|
| latin | 68839 |
| han | 25462 |
| arabic | 3992 |
| cyrillic | 3949 |
| hangul | 3564 |
| hebrew | 3164 |
| thai | 2570 |
| kana | 2067 |
| other | 1231 |
| greek | 193 |
| armenian | 77 |
| devanagari | 63 |
| bengali | 44 |
| myanmar | 37 |
| georgian | 37 |
| khmer | 36 |
| lao | 33 |
| malayalam | 32 |
| tamil | 28 |
| sinhala | 26 |
| kannada | 25 |
| mixed | 19 |
| gujarati | 16 |
| telugu | 14 |
| gurmukhi | 10 |
| oriya | 7 |

## 2. Tier 1: the character floor

Each language gets every character of its standard lists, with no measurement (S2.3). Tier 1 comes from the standard lists
only and is never extended from data.

| Applies to | Standard list |
|---|---|
| All languages | CLDR exemplar characters (`exemplarCharacters`), punctuation (`punctuation`), and the digits of the default and native (`native`) numbering systems. Upper case too, where the script has case |
| Indic, Southeast Asian and Georgian scripts | In addition, every assigned character of the script's Unicode block, plus ZWJ and ZWNJ |
| Korean | Hangul from the 2,350 KS X 1001 syllables |
| Japanese | Kanji from JIS X 0208 levels 1 and 2 |
| Simplified Chinese | Hanzi from GB 2312 |
| Traditional Chinese, Cantonese | Hanzi from the 5,401 Big5 common characters (常用字) |

A language missing from CLDR takes its parent language's set, noted in the table. Characters that are already single Qwen3
tokens are not added again. Non-standard characters found in spoken data are kept aside as candidates for tiers 2 and 3.

In [3]:
CLDR_LOCALE = {
    "English": "en", "Norwegian Bokmal": "nb", "Norwegian Nynorsk": "nn", "Persian": "fa", "Dari": "fa-AF",
    "Simplified Chinese": "zh", "Traditional Chinese": "zh-Hant", "Cantonese": "yue",
    "Standard Arabic": "ar", "Najdi Arabic": "ar-SA", "Levantine Arabic": "apc", "Egyptian Arabic": "ar-EG",
    "Moroccan Arabic": "ar-MA", "Mesopotamian Arabic": "ar-IQ", "Taizzi-Adeni Arabic": "ar-YE", "Tunisian Arabic": "ar-TN",
    "Tagalog": "fil", "Malay": "ms", "Serbian": "sr", "Tosk Albanian": "sq", "Eastern Yiddish": "yi",
    "Northern Uzbek": "uz", "North Azerbaijani": "az", "Latvian": "lv", "Estonian": "et", "Nepali": "ne", "Swahili": "sw",
    "Bhojpuri": "bho", "Maithili": "mai", "Kabuverdianu": "kea", "Sardinian": "sc",
}
# Languages CLDR does not describe, read from the closest language it does.
CLDR_PARENT = {
    "Chhattisgarhi": "hi", "Awadhi": "hi", "Magahi": "hi", "Venetian": "it", "Lombard": "it", "Ligurian": "it",
    "Sicilian": "it", "Friulian": "it", "Silesian": "pl", "Limburgish": "nl", "Minangkabau": "id", "Balinese": "id",
    "Banjar": "id", "Pangasinan": "fil", "Iloko": "fil", "Waray": "fil", "Papiamento": "es", "Tok Pisin": "en",
    "Haitian": "fr", "Tatar": "ru", "Bashkir": "ru",
}

cache = Path(".cache/cldr") / CLDR_RELEASE
cache.mkdir(parents=True, exist_ok=True)


def cldr_file(package, locale, name):
    path = cache / package / f"{locale}.{name}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        url = (f"https://raw.githubusercontent.com/unicode-org/cldr-json/{CLDR_RELEASE}/cldr-json/{package}/"
               + (f"main/{locale}/{name}.json" if locale else f"{name}.json"))
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                path.write_bytes(response.read())
        except urllib.error.HTTPError:
            return None
    return json.load(open(path))


def cldr_characters(locale):
    data = cldr_file("cldr-misc-full", locale, "characters")
    return None if data is None else data["main"][locale]["characters"]


NUMBERING_SYSTEMS = cldr_file("cldr-core", None, "supplemental/numberingSystems")["supplemental"]["numberingSystems"]


def cldr_digits(locale):
    data = cldr_file("cldr-numbers-full", locale, "numbers")
    if data is None:
        return set()
    numbers = data["main"][locale]["numbers"]
    systems = {numbers["defaultNumberingSystem"], *numbers.get("otherNumberingSystems", {}).values()}
    return {digit for system in systems for digit in NUMBERING_SYSTEMS.get(system, {}).get("_digits", "")}


def expand_exemplars(text):
    # CLDR writes combining marks and some symbols as \uXXXX escapes, and quotes other characters with a backslash.
    text = re.sub(r"\\u([0-9A-Fa-f]{4})", lambda match: chr(int(match.group(1), 16)), text).replace("\\", "")
    items = []
    for part in text.strip("[]").split():
        if part.startswith("{") and part.endswith("}"):
            items.append(part[1:-1])
        elif len(part) == 3 and part[1] == "-":
            items += [chr(code) for code in range(ord(part[0]), ord(part[2]) + 1)]
        else:
            items += list(part)
    with_case = set(items)
    with_case |= {item.upper() for item in items if item.upper() != item}
    return with_case


standard = {}
source = {}
for language in QWEN3_LANGUAGES:
    if language.name == "Korean":
        continue
    candidates = [CLDR_LOCALE.get(language.name), language.wikipedia, CLDR_PARENT.get(language.name)]
    for locale in [code for code in candidates if code]:
        characters = cldr_characters(locale)
        if characters is not None:
            standard[language.name] = (expand_exemplars(characters["exemplarCharacters"])
                                       | expand_exemplars(characters.get("punctuation", ""))
                                       | cldr_digits(locale))
            source[language.name] = locale if locale != CLDR_PARENT.get(language.name) else f"{locale} (related language)"
            break
missing_cldr = [language.name for language in QWEN3_LANGUAGES if language.name not in standard and language.name != "Korean"]
print("Languages with no CLDR exemplar characters:", missing_cldr)

Languages with no CLDR exemplar characters: []


### Korean: KS X 1001

The Korean spoken data is the v1 data: every utterance of the AI Hub daily conversation multi-session corpus (011) and the
Korean source text of the AI Hub colloquial Korean-English corpus.

In [4]:
KOREAN_SPOKEN = Path(".cache/datasets")


def korean_spoken_texts():
    for path in sorted((KOREAN_SPOKEN / "multisession_daily").rglob("*.json")):
        for session in json.load(open(path))["sessionInfo"]:
            for turn in session["dialog"]:
                yield turn["utterance"]
    for path in sorted((KOREAN_SPOKEN / "colloquial_koen").rglob("*한영*.json")):
        data = json.load(open(path))
        for row in data["data"] if isinstance(data, dict) else data:
            yield row["ko_original"]


korean_texts = list(korean_spoken_texts())
ks_x_1001 = {chr(code) for code in range(0xAC00, 0xD7A4) if len(chr(code).encode("euc-kr", "ignore")) == 2}
syllables_seen = collections.Counter(ch for text in korean_texts for ch in text if 0xAC00 <= ord(ch) <= 0xD7A3)
outside = {ch: count for ch, count in syllables_seen.items() if ch not in ks_x_1001}
korean_cldr = cldr_characters("ko")
standard["Korean"] = ks_x_1001 | expand_exemplars(korean_cldr["punctuation"]) | cldr_digits("ko")
# Syllables the spoken data uses outside KS X 1001 are candidates for tiers 2 and 3, not tier 1.
json.dump(outside, open(OUT / "korean_candidates_outside_ks.json", "w"), ensure_ascii=False, indent=1)
source["Korean"] = "KS X 1001 + CLDR punctuation and digits"
print(f"{len(korean_texts):,} spoken sentences, {len(syllables_seen):,} distinct syllables, {len(outside)} of them outside KS X 1001")
print("Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first:", sorted(outside.items(), key=lambda kv: -kv[1])[:30])

3,273,481 spoken sentences, 2,501 distinct syllables, 424 of them outside KS X 1001
Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first: [('됬', 1500), ('몐', 390), ('떄', 185), ('슌', 142), ('귬', 111), ('녓', 61), ('춉', 55), ('웻', 52), ('뱡', 46), ('먄', 43), ('곘', 41), ('똠', 40), ('븡', 37), ('렜', 36), ('픙', 36), ('녤', 34), ('댠', 33), ('됭', 33), ('퀜', 30), ('똣', 26), ('헀', 24), ('닏', 22), ('솁', 22), ('쎼', 21), ('늗', 21), ('럅', 20), ('뢱', 20), ('잌', 19), ('탼', 19), ('졋', 18)]


### Tier 1 tokens per language

A language's tier 1 tokens are its standard characters that are not single tokens in the Qwen3 vocabulary, and the word-start
form (`▁` plus the character, S12.1b) of every standard letter and digit that has no space-led form in Qwen3 or Helium.
Without that form a word starting with the character spends a frame on `▁` alone. Word-start forms are written here with a
leading space, as the other surface strings are.

In [5]:
# Scripts whose CLDR lists leave out vowel signs, letters with a dot or old letters, so the whole Unicode block is taken.
BLOCK_SCRIPTS = {
    "devanagari": (0x0900, 0x097F), "bengali": (0x0980, 0x09FF), "gurmukhi": (0x0A00, 0x0A7F), "gujarati": (0x0A80, 0x0AFF),
    "oriya": (0x0B00, 0x0B7F), "tamil": (0x0B80, 0x0BFF), "telugu": (0x0C00, 0x0C7F), "kannada": (0x0C80, 0x0CFF),
    "malayalam": (0x0D00, 0x0D7F), "sinhala": (0x0D80, 0x0DFF), "thai": (0x0E00, 0x0E7F), "lao": (0x0E80, 0x0EFF),
    "myanmar": (0x1000, 0x109F), "georgian": (0x10A0, 0x10FF), "khmer": (0x1780, 0x17FF),
}


# Zero width joiner and non-joiner, which these scripts' orthographies use between letters.
JOINERS = {"\u200c", "\u200d"}


def block_letters(chars):
    letters = set()
    for script in {script_of(ch) for ch in chars if len(ch) == 1 and ch.isalpha()} & set(BLOCK_SCRIPTS):
        start, end = BLOCK_SCRIPTS[script]
        letters |= {chr(code) for code in range(start, end + 1) if unicodedata.name(chr(code), "")}
        letters |= JOINERS
    return letters


def encoding_hanzi(codec, keep=lambda encoded: True):
    hanzi = set()
    for code in list(range(0x4E00, 0xA000)) + list(range(0x3400, 0x4DC0)) + list(range(0xF900, 0xFB00)):
        try:
            encoded = chr(code).encode(codec)
        except UnicodeEncodeError:
            continue
        if keep(encoded):
            hanzi.add(chr(code))
    return hanzi


def big5_common(encoded):
    # Big5 level 1 (常用字) runs from A440 to C67E.
    return len(encoded) == 2 and (0xA4 <= encoded[0] < 0xC6 or (encoded[0] == 0xC6 and encoded[1] <= 0x7E))


HANZI_STANDARD = {
    "Japanese": ("JIS X 0208", encoding_hanzi("euc_jp", lambda encoded: len(encoded) == 2)),
    "Simplified Chinese": ("GB 2312", encoding_hanzi("gb2312")),
    "Traditional Chinese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
    "Cantonese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
}
for name, (label, hanzi) in HANZI_STANDARD.items():
    # The encoding standard replaces CLDR's ideographs; kana, punctuation and digits stay as CLDR lists them.
    standard[name] = {ch for ch in standard[name] if script_of(ch) != "han"} | hanzi
    source[name] = f"{source[name]}, Han characters from {label}"
for language in QWEN3_LANGUAGES:
    extra = block_letters(standard.get(language.name, set()))
    if extra - standard.get(language.name, set()):
        standard[language.name] = standard[language.name] | extra
        source[language.name] = f"{source[language.name]} + Unicode block"

spaced = set(qwen_surface) | set(helium_surface)
tier1 = {}
rows = []
for language in QWEN3_LANGUAGES:
    chars = standard.get(language.name, set())
    added = sorted(ch for ch in chars if ch not in qwen_surface)
    starts = sorted(" " + ch for ch in chars if len(ch) == 1 and (ch.isalpha() or ch.isdigit()) and " " + ch not in spaced)
    tier1[language.name] = added + starts
    rows.append([language.name, source.get(language.name, "none"), len(chars), len(chars) - len(added), len(added), len(starts)])
table(rows, ["Language", "Standard source", "Standard characters", "Already in Qwen3", "Added as tier 1", "Word-start forms added"])
all_tier1 = sorted({ch for chars in tier1.values() for ch in chars})
print(f"Tier 1 tokens summed over languages: {sum(len(v) for v in tier1.values()):,}, distinct: {len(all_tier1):,}")
json.dump({"cldr_release": CLDR_RELEASE, "source": source, "tier1": tier1}, open(OUT / "tier1.json", "w"), ensure_ascii=False, indent=1)

| Language | Standard source | Standard characters | Already in Qwen3 | Added as tier 1 | Word-start forms added |
|---|---|---|---|---|---|
| English | en | 94 | 94 | 0 | 10 |
| French | fr | 124 | 124 | 0 | 23 |
| Portuguese | pt | 120 | 120 | 0 | 19 |
| German | de | 102 | 102 | 0 | 11 |
| Romanian | ro | 99 | 99 | 0 | 16 |
| Swedish | sv | 104 | 104 | 0 | 10 |
| Danish | da | 98 | 98 | 0 | 11 |
| Bulgarian | bg | 98 | 98 | 0 | 19 |
| Russian | ru | 108 | 108 | 0 | 23 |
| Czech | cs | 118 | 118 | 0 | 28 |
| Greek | el | 104 | 101 | 3 | 50 |
| Ukrainian | uk | 104 | 104 | 0 | 25 |
| Spanish | es | 112 | 112 | 0 | 15 |
| Dutch | nl | 116 | 114 | 2 | 17 |
| Slovak | sk | 126 | 123 | 3 | 28 |
| Croatian | hr | 97 | 94 | 3 | 14 |
| Polish | pl | 111 | 111 | 0 | 21 |
| Lithuanian | lt | 94 | 93 | 1 | 24 |
| Norwegian Bokmal | nb | 110 | 110 | 0 | 14 |
| Norwegian Nynorsk | nn | 110 | 110 | 0 | 14 |
| Persian | fa | 75 | 62 | 13 | 19 |
| Slovenian | sl | 84 | 84 | 0 | 12 |
| Gujarati | gu + Unicode block | 135 | 59 | 76 | 73 |
| Latvian | lv | 110 | 110 | 0 | 28 |
| Italian | it | 102 | 102 | 0 | 16 |
| Occitan | oc | 115 | 115 | 0 | 17 |
| Nepali | ne + Unicode block | 149 | 81 | 68 | 86 |
| Marathi | mr + Unicode block | 161 | 91 | 70 | 86 |
| Belarusian | be | 94 | 91 | 3 | 22 |
| Serbian | sr | 93 | 93 | 0 | 22 |
| Luxembourgish | lb | 100 | 100 | 0 | 12 |
| Venetian | vec | 108 | 107 | 1 | 17 |
| Assamese | as + Unicode block | 134 | 80 | 54 | 65 |
| Welsh | cy | 156 | 153 | 3 | 41 |
| Silesian | szl | 115 | 115 | 0 | 23 |
| Asturian | ast | 110 | 109 | 1 | 19 |
| Chhattisgarhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Awadhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Maithili | mai + Unicode block | 180 | 108 | 72 | 96 |
| Bhojpuri | bho + Unicode block | 172 | 104 | 68 | 96 |
| Sindhi | sd | 82 | 64 | 18 | 33 |
| Irish | ga | 88 | 88 | 0 | 13 |
| Faroese | fo | 98 | 98 | 0 | 17 |
| Hindi | hi + Unicode block | 171 | 103 | 68 | 96 |
| Punjabi | pa + Unicode block | 119 | 48 | 71 | 71 |
| Bengali | bn + Unicode block | 134 | 80 | 54 | 65 |
| Oriya | or + Unicode block | 139 | 53 | 86 | 73 |
| Tajik | tg | 113 | 110 | 3 | 29 |
| Eastern Yiddish | yi | 74 | 61 | 13 | 16 |
| Lombard | lmo | 87 | 87 | 0 | 10 |
| Ligurian | lij | 132 | 132 | 0 | 27 |
| Sicilian | scn | 108 | 108 | 0 | 21 |
| Friulian | fur | 99 | 99 | 0 | 19 |
| Sardinian | sc | 92 | 92 | 0 | 16 |
| Galician | gl | 114 | 114 | 0 | 17 |
| Catalan | ca | 119 | 119 | 0 | 17 |
| Icelandic | is | 106 | 106 | 0 | 18 |
| Tosk Albanian | sq | 115 | 111 | 4 | 12 |
| Limburgish | nl (related language) | 116 | 114 | 2 | 17 |
| Dari | fa-AF | 75 | 62 | 13 | 19 |
| Afrikaans | af | 116 | 116 | 0 | 17 |
| Macedonian | mk | 95 | 95 | 0 | 24 |
| Sinhala | si + Unicode block | 135 | 69 | 66 | 79 |
| Urdu | ur | 70 | 60 | 10 | 32 |
| Magahi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Bosnian | bs | 96 | 93 | 3 | 14 |
| Armenian | hy | 98 | 90 | 8 | 86 |
| Simplified Chinese | zh, Han characters from GB 2312 | 6828 | 6705 | 123 | 6727 |
| Traditional Chinese | zh-Hant, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Cantonese | yue, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Burmese | my + Unicode block | 182 | 58 | 124 | 94 |
| Standard Arabic | ar | 84 | 71 | 13 | 26 |
| Najdi Arabic | ar-SA | 74 | 61 | 13 | 16 |
| Levantine Arabic | apc | 76 | 73 | 3 | 16 |
| Egyptian Arabic | ar-EG | 74 | 61 | 13 | 16 |
| Moroccan Arabic | ar-MA | 84 | 71 | 13 | 26 |
| Mesopotamian Arabic | ar-IQ | 74 | 61 | 13 | 16 |
| Taizzi-Adeni Arabic | ar-YE | 74 | 61 | 13 | 16 |
| Tunisian Arabic | ar-TN | 84 | 71 | 13 | 26 |
| Hebrew | he | 58 | 58 | 0 | 15 |
| Maltese | mt | 98 | 96 | 2 | 22 |
| Indonesian | id | 94 | 94 | 0 | 10 |
| Malay | ms | 94 | 94 | 0 | 10 |
| Tagalog | fil | 95 | 95 | 0 | 12 |
| Cebuano | ceb | 74 | 74 | 0 | 10 |
| Javanese | jv | 93 | 83 | 10 | 27 |
| Sundanese | su | 96 | 96 | 0 | 10 |
| Minangkabau | id (related language) | 94 | 94 | 0 | 10 |
| Balinese | id (related language) | 94 | 94 | 0 | 10 |
| Banjar | id (related language) | 94 | 94 | 0 | 10 |
| Pangasinan | fil (related language) | 95 | 95 | 0 | 12 |
| Iloko | fil (related language) | 95 | 95 | 0 | 12 |
| Waray | fil (related language) | 95 | 95 | 0 | 12 |
| Tamil | ta + Unicode block | 116 | 71 | 45 | 57 |
| Telugu | te + Unicode block | 134 | 47 | 87 | 77 |
| Kannada | kn + Unicode block | 135 | 68 | 67 | 77 |
| Malayalam | ml + Unicode block | 165 | 77 | 88 | 87 |
| Turkish | tr | 100 | 100 | 0 | 13 |
| North Azerbaijani | az | 106 | 106 | 0 | 15 |
| Northern Uzbek | uz | 99 | 95 | 4 | 11 |
| Kazakh | kk | 126 | 126 | 0 | 40 |
| Bashkir | ba | 126 | 125 | 1 | 41 |
| Tatar | tt | 118 | 117 | 1 | 35 |
| Thai | th + Unicode block | 125 | 110 | 15 | 69 |
| Lao | lo + Unicode block | 127 | 76 | 51 | 76 |
| Finnish | fi | 96 | 96 | 0 | 11 |
| Estonian | et | 92 | 92 | 0 | 13 |
| Hungarian | hu | 141 | 120 | 21 | 17 |
| Vietnamese | vi | 220 | 220 | 0 | 120 |
| Khmer | km + Unicode block | 145 | 64 | 81 | 74 |
| Japanese | ja, Han characters from JIS X 0208 | 6627 | 4311 | 2316 | 6501 |
| Korean | KS X 1001 + CLDR punctuation and digits | 2443 | 2165 | 278 | 2154 |
| Georgian | ka + Unicode block | 169 | 84 | 85 | 130 |
| Basque | eu | 98 | 98 | 0 | 12 |
| Haitian | ht | 101 | 101 | 0 | 12 |
| Papiamento | pap | 99 | 99 | 0 | 16 |
| Kabuverdianu | kea | 98 | 97 | 1 | 12 |
| Tok Pisin | tpi | 67 | 67 | 0 | 10 |
| Swahili | sw | 74 | 74 | 0 | 10 |

Tier 1 tokens summed over languages: 37,299, distinct: 18,372


### Check: do byte fragments drop to 0% on spoken Korean?

Spoken sentences are cut with the unchanged Qwen3 tokenizer, and each byte-fragment token (a token that is not a character on
its own) is traced to the character it came from. Fragments are split by position: the first character of a word (after a
space) or a character inside a word. Any fragment tier 1 does not cover is examined before deciding.

In [6]:
covered = set(qwen_surface) | set(tier1["Korean"])
fragment_tokens = total_tokens = 0
causes = collections.Counter()
for text in korean_texts[:20000]:
    encoding = qwen(text, add_special_tokens=False, return_offsets_mapping=True)
    total_tokens += len(encoding["input_ids"])
    for token_id, (start, end) in zip(encoding["input_ids"], encoding["offset_mapping"]):
        if "�" not in qwen.decode([token_id]):
            continue
        fragment_tokens += 1
        position = start
        while position < len(text) and text[position] == " ":
            position += 1
        if position < len(text):
            causes[(text[position], position > 0 and text[position - 1] == " ")] += 1
print(f"Qwen3 as is: byte-fragment tokens {fragment_tokens / total_tokens:.1%} ({fragment_tokens:,}/{total_tokens:,}), 20,000 sentences")
rows = []
for label, word_start in (("first character of a word", True), ("character inside a word", False)):
    forms = {ch: count for (ch, start), count in causes.items() if start == word_start}
    in_qwen = sum(count for ch, count in forms.items() if ch in qwen_surface)
    in_tier1 = sum(count for ch, count in forms.items() if ch not in qwen_surface and ch in covered)
    rest = sum(count for ch, count in forms.items() if ch not in covered)
    rows.append([label, sum(forms.values()), in_qwen, in_tier1, rest])
table(rows, ["Fragment position", "Fragments", "Character is a single Qwen3 token", "Character is in tier 1", "Not covered"])

Qwen3 as is: byte-fragment tokens 14.5% (84,206/580,161), 20,000 sentences


| Fragment position | Fragments | Character is a single Qwen3 token | Character is in tier 1 | Not covered |
|---|---|---|---|---|
| first character of a word | 80663 | 80626 | 4 | 33 |
| character inside a word | 3543 | 3358 | 14 | 171 |

## 3. Tokenizer comparison: byte BPE and Unigram with byte fallback

The same token list is used to cut text in two ways (S12.1b).

- **Qwen3 byte BPE**: the Qwen3 tokenizer as is. Each word is cut with a leading space, as the duplex stream cuts words.
- **Unigram**: Qwen3 and Helium token strings, the leading space written as `▁` (Qwen3's `Ġ` becomes `▁`, S12.1b). A string
  both have is one token, Helium's. `▁x` and `x` stay two tokens. Byte-fragment tokens are dropped and replaced by
  `byte_fallback` (256 byte tokens). Every word is cut with `▁` in front (`add_dummy_prefix`), as `text_stream` cuts a word
  with a leading space, so the token list marks every word start whether or not an `EPAD` precedes it.
- **Unigram + tier 1**: the above plus the tier 1 tokens of every language (characters and word-start forms).

The processor turns input text into NFC before cutting it (S12.1), so every token enters as its NFC form and tokens that become
the same are merged. Vietnamese `ấ` written as a base letter plus a combining mark joins the composed `ấ`, and Devanagari `क़`,
which NFC splits into `क` and a nukta, stays one token as that two-character string. Helium tokens keep their form for their ids.

Unigram scores come from EM: cut the training corpus into its most likely segmentation (Viterbi) with the fixed token list,
re-score each token by how often it was used, and repeat. The training corpus is spoken training data (the 011 training split
and the Korean and English of the colloquial Korean-English training split). The comparison uses 3,000 Korean and 3,000
English sentences from the colloquial Korean-English validation split, which does not overlap the training data.

In [7]:
import math
import random

import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as model_pb2


def colloquial(split_dir, field, limit=None):
    texts = []
    for path in sorted((KOREAN_SPOKEN / "colloquial_koen").glob(f"{split_dir}/*한영*.json")):
        data = json.load(open(path))
        texts += [row[field] for row in (data["data"] if isinstance(data, dict) else data)]
    return texts[:limit] if limit else texts


def multisession_training():
    for path in sorted((KOREAN_SPOKEN / "multisession_daily" / "Training").rglob("*.json")):
        for session in json.load(open(path))["sessionInfo"]:
            for turn in session["dialog"]:
                yield turn["utterance"]


eval_ko = colloquial("VL1", "ko_original", 3000)
eval_en = colloquial("VL1", "en", 3000)
random.seed(0)
train_ko = random.sample(list(multisession_training()), 150_000) + random.sample(colloquial("TL2", "ko_original"), 150_000)
train_en = random.sample(colloquial("TL2", "en"), 300_000)
train_words = [word for text in train_ko + train_en for word in text.split()]
print(f"EM training words: {len(train_words):,}, evaluation sentences: Korean {len(eval_ko):,}, English {len(eval_en):,}")

EM training words: 5,936,282, evaluation sentences: Korean 3,000, English 3,000


In [8]:
def as_piece(surface):
    # A leading space is SentencePiece's word-start mark.
    return "▁" + surface[1:] if surface.startswith(" ") else surface


def unigram_pieces(extra=()):
    # Qwen3 and Helium strings as pieces; a string both have is one piece.
    pieces = set()
    for surface in list(qwen_surface) + list(helium_surface) + list(extra):
        if "�" in surface:
            continue
        piece = as_piece(surface)
        # Input text is NFC, so a piece enters as its NFC form; Helium's are kept as they are for their ids.
        if surface not in helium_surface:
            piece = unicodedata.normalize("NFC", piece)
        # SentencePiece takes no whitespace or control character inside a piece.
        if piece and not any(ch.isspace() or unicodedata.category(ch) == "Cc" for ch in piece):
            pieces.add(piece)
    return sorted(pieces)


def build_unigram(pieces, scores):
    model = model_pb2.ModelProto()
    model.trainer_spec.model_type = model_pb2.TrainerSpec.UNIGRAM
    model.trainer_spec.byte_fallback = True
    model.trainer_spec.unk_id, model.trainer_spec.bos_id, model.trainer_spec.eos_id, model.trainer_spec.pad_id = 0, 1, 2, -1
    model.normalizer_spec.name = "identity"
    model.normalizer_spec.add_dummy_prefix = True
    model.normalizer_spec.remove_extra_whitespaces = False
    model.normalizer_spec.escape_whitespaces = True
    for piece, kind in (("<unk>", model_pb2.ModelProto.SentencePiece.UNKNOWN), ("<s>", model_pb2.ModelProto.SentencePiece.CONTROL),
                        ("</s>", model_pb2.ModelProto.SentencePiece.CONTROL)):
        model.pieces.add(piece=piece, score=0.0, type=kind)
    for byte in range(256):
        model.pieces.add(piece=f"<0x{byte:02X}>", score=0.0, type=model_pb2.ModelProto.SentencePiece.BYTE)
    for piece in pieces:
        model.pieces.add(piece=piece, score=scores[piece], type=model_pb2.ModelProto.SentencePiece.NORMAL)
    return spm.SentencePieceProcessor(model_proto=model.SerializeToString())


def fit_unigram(pieces, words, iterations=4, verbose=True):
    # Hard EM on a fixed vocabulary: cut with the current scores, then score each piece by how often it was used.
    # `words` is a list of words or a Counter of them.
    scores = {piece: -math.log(len(pieces)) for piece in pieces}
    word_counts = words if isinstance(words, collections.Counter) else collections.Counter(words)
    for iteration in range(iterations):
        model = build_unigram(pieces, scores)
        counts = collections.Counter()
        words_in_order = list(word_counts)
        for word, pieces_of_word in zip(words_in_order, model.encode(words_in_order, out_type=str, num_threads=THREADS)):
            count = word_counts[word]
            for piece in pieces_of_word:
                counts[piece] += count
        total = sum(counts[piece] for piece in pieces) + len(pieces) * 0.1
        scores = {piece: math.log((counts[piece] + 0.1) / total) for piece in pieces}
        used = sum(1 for piece in pieces if counts[piece])
        if verbose:
            print(f"  EM {iteration + 1}: tokens used {used:,}/{len(pieces):,}")
    return build_unigram(pieces, scores)


def measure_unigram(model, texts):
    tokens = fragments = words = 0
    for text in texts:
        for word in text.split():
            pieces = model.encode(word, out_type=str)
            words += 1
            tokens += len(pieces)
            fragments += sum(1 for piece in pieces if piece.startswith("<0x"))
    return tokens / words, fragments / tokens


def measure_bpe(texts):
    tokens = fragments = words = 0
    for text in texts:
        for word in text.split():
            ids = qwen.encode(" " + word, add_special_tokens=False)
            words += 1
            tokens += len(ids)
            fragments += sum(1 for i in ids if "�" in qwen.decode([i]))
    return tokens / words, fragments / tokens


base_pieces = unigram_pieces()
tier1_all = sorted({ch for chars in tier1.values() for ch in chars})
print(f"Unigram tokens: {len(base_pieces):,} from Qwen3 and Helium ({len(qwen_surface):,} and {len(helium_surface):,} strings), tier 1: {len(tier1_all):,}")
print("Unigram EM on Qwen3 and Helium tokens")
unigram_qwen = fit_unigram(base_pieces, train_words)
print("Unigram EM on Qwen3 and Helium tokens + tier 1")
unigram_tier1 = fit_unigram(unigram_pieces(tier1_all), train_words)

rows = []
for name, measure in (("Qwen3 byte BPE", measure_bpe), ("Unigram (Qwen3 and Helium tokens)", lambda t: measure_unigram(unigram_qwen, t)),
                      ("Unigram (Qwen3 and Helium tokens + tier 1)", lambda t: measure_unigram(unigram_tier1, t))):
    ko_tokens, ko_fragments = measure(eval_ko)
    en_tokens, en_fragments = measure(eval_en)
    rows.append([name, f"{ko_tokens:.2f}", f"{ko_fragments:.2%}", f"{en_tokens:.2f}", f"{en_fragments:.2%}"])
table(rows, ["Tokenizer", "Korean tokens per word", "Korean byte fragments", "English tokens per word", "English byte fragments"])
elapsed("tokenizer comparison")


Unigram tokens: 151,634 from Qwen3 and Helium (150,303 and 31,739 strings), tier 1: 18,372
Unigram EM on Qwen3 and Helium tokens


  EM 1: tokens used 45,284/151,634


  EM 2: tokens used 43,313/151,634


  EM 3: tokens used 43,079/151,634


  EM 4: tokens used 43,016/151,634


Unigram EM on Qwen3 and Helium tokens + tier 1


... (2 outputs left out)

  EM 3: tokens used 44,843/169,970


  EM 4: tokens used 44,782/169,970


| Tokenizer | Korean tokens per word | Korean byte fragments | English tokens per word | English byte fragments |
|---|---|---|---|---|
| Qwen3 byte BPE | 2.76 | 15.70% | 1.26 | 0.00% |
| Unigram (Qwen3 and Helium tokens) | 2.70 | 0.00% | 1.26 | 0.00% |
| Unigram (Qwen3 and Helium tokens + tier 1) | 2.51 | 0.00% | 1.25 | 0.00% |

[tokenizer comparison] 254 s since the start


## 4. Spoken measurement per language (Unigram with tier 1)

Each language uses whatever spoken data exists (S12.6): FLEURS transcripts (with utterance durations) and OpenSubtitles
(spoken subtitles, up to 200,000 lines streamed from the start of the language file; the tier 2 candidates come from them). Only languages with neither fall back to
Wikipedia sentences, and the table says so. Lines with control or private-use characters are broken data and are dropped.

Measured values:
- **Byte fragments**: the share of byte tokens. Tier 1 holds the language's characters, so this should be 0% (S2.3).
- **Tokens per word**: tokens per whitespace-separated word. For languages written without spaces (Chinese, Japanese, Thai
  and others) a whitespace unit is not a word, so tokens per character is shown as well.
- **Tokens per second, frame occupancy** (languages with FLEURS): utterance tokens divided by utterance duration (samples /
  16,000). Occupancy is (tokens + one `EPAD` per word) / (duration × 12.5 frames). Above 1, the utterance does not fit its
  frames. This approximates over whole utterances because word timings are missing; the per-word cut rate (S2.2) is
  measured separately on data with word timings.

The Unigram scores here were fitted on Korean and English only (section 3), so tokens of other languages all sit at the floor
score. Their tokens per word can therefore come out higher than with the final scores. The final scores are refitted on a mix
of spoken data from every language (S12.1).

In [9]:
import gzip
import io

from huggingface_hub import hf_hub_download

OPENSUBTITLES = "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/mono/{code}.txt.gz"
OPUS_CODE = {"Simplified Chinese": "zh_cn", "Traditional Chinese": "zh_tw", "Portuguese": "pt_br", "Norwegian Bokmal": "no",
             "Tagalog": "tl", "Malay": "ms"}
spoken_cache = OUT / "spoken"
spoken_cache.mkdir(exist_ok=True)


def fleurs_utterances(code):
    rows = []
    for split in ("train", "dev", "test"):
        path = hf_hub_download("google/fleurs", f"data/{code}/{split}.tsv", repo_type="dataset")
        for line in open(path, encoding="utf-8"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 6:
                rows.append((fields[2], int(fields[5]) / 16000))
    return rows


OPENSUBTITLES_LINES = 200_000


def opensubtitles_lines(code, limit=OPENSUBTITLES_LINES):
    path = spoken_cache / f"opensubtitles_{code}_{limit}.txt"
    if not path.exists():
        lines = []
        try:
            with urllib.request.urlopen(OPENSUBTITLES.format(code=code), timeout=120) as response:
                with gzip.open(response, "rt", encoding="utf-8", errors="replace") as stream:
                    for line in stream:
                        line = line.strip()
                        if line:
                            lines.append(line)
                        if len(lines) >= limit:
                            break
        except urllib.error.HTTPError:
            lines = []
        path.write_text("\n".join(lines), encoding="utf-8")
    text = path.read_text(encoding="utf-8")
    return text.splitlines() if text else []


def wikipedia_sentences(code, limit=30_000):
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi

    files = sorted(f.path for f in HfApi().list_repo_tree("wikimedia/wikipedia", path_in_repo=f"20231101.{code}", repo_type="dataset"))
    path = hf_hub_download("wikimedia/wikipedia", files[0], repo_type="dataset")
    sentences = []
    for batch in pq.ParquetFile(path).iter_batches(columns=["text"]):
        for text in batch.column("text").to_pylist():
            sentences += [line.strip() for line in text.split("\n") if len(line.strip()) > 20]
            if len(sentences) >= limit:
                return sentences[:limit]
    return sentences


def measure_texts(texts):
    tokens = fragments = words = characters = 0
    all_words = [word for text in texts for word in text.split()]
    for word, pieces in zip(all_words, unigram_tier1.encode(all_words, out_type=str, num_threads=THREADS)):
        words += 1
        characters += len(word)
        tokens += len(pieces)
        fragments += sum(1 for piece in pieces if piece.startswith("<0x"))
    return {"tokens": tokens, "words": words, "byte_fragments": fragments / max(tokens, 1),
            "tokens_per_word": tokens / max(words, 1), "tokens_per_character": tokens / max(characters, 1)}


def measure_timing(utterances):
    seconds = tokens = epads = 0
    overflow = 0
    for text, duration in utterances:
        words = text.split()
        count = sum(len(unigram_tier1.encode(word)) for word in words)
        seconds += duration
        tokens += count
        epads += len(words)
        overflow += (count + len(words)) > duration * 12.5
    return {"tokens_per_second": tokens / seconds, "occupancy": (tokens + epads) / (seconds * 12.5),
            "overflow_utterances": overflow / max(len(utterances), 1)}


measurements = {}
spoken = {}
for language in QWEN3_LANGUAGES:
    texts, sources, timing, durations = [], [], None, {}
    if language.fleurs:
        utterances = fleurs_utterances(language.fleurs)
        texts += [text for text, _ in utterances]
        sources.append(f"FLEURS {len(utterances):,}")
        timing = measure_timing(utterances)
        durations = dict(utterances)
    opus = OPUS_CODE.get(language.name, language.wikipedia)
    if opus:
        lines = opensubtitles_lines(opus)
        if lines:
            texts += lines
            sources.append(f"OpenSubtitles {len(lines):,}")
    if not texts and language.wikipedia:
        sentences = wikipedia_sentences(language.wikipedia)
        texts += sentences
        sources.append(f"Wikipedia {len(sentences):,} (not spoken)")
    # Lines carrying control or private-use characters are broken data, not language.
    texts = [text for text in texts if not any(unicodedata.category(ch) in ("Cc", "Co") for ch in text)]
    result = measure_texts(texts) if texts else {}
    spoken[language.name] = {"texts": texts, "durations": durations}
    result["sources"] = ", ".join(sources) or "none"
    if timing:
        result.update(timing)
    measurements[language.name] = result

json.dump(measurements, open(OUT / "spoken_measurements.json", "w"), ensure_ascii=False, indent=1)
rows = []
for language in QWEN3_LANGUAGES:
    m = measurements[language.name]
    rows.append([
        language.name, m["sources"],
        f"{m['byte_fragments']:.2%}" if "byte_fragments" in m else "-",
        f"{m['tokens_per_word']:.2f}" if "tokens_per_word" in m else "-",
        f"{m['tokens_per_character']:.2f}" if "tokens_per_character" in m else "-",
        f"{m['tokens_per_second']:.2f}" if "tokens_per_second" in m else "-",
        f"{m['occupancy']:.2f}" if "occupancy" in m else "-",
        f"{m['overflow_utterances']:.1%}" if "overflow_utterances" in m else "-",
    ])
table(rows, ["Language", "Data", "Byte fragments", "Tokens per word", "Tokens per character", "Tokens per second", "Frame occupancy", "Overflowing utterances"])
elapsed("spoken measurement")


| Language | Data | Byte fragments | Tokens per word | Tokens per character | Tokens per second | Frame occupancy | Overflowing utterances |
|---|---|---|---|---|---|---|---|
| English | FLEURS 3,643, OpenSubtitles 200,000 | 0.00% | 1.34 | 0.31 | 2.57 | 0.37 | 0.0% |
| French | FLEURS 4,158, OpenSubtitles 200,000 | 0.00% | 1.75 | 0.38 | 3.60 | 0.46 | 0.0% |
| Portuguese | FLEURS 4,098, OpenSubtitles 200,000 | 0.00% | 1.80 | 0.38 | 2.98 | 0.38 | 0.0% |
| German | FLEURS 4,212, OpenSubtitles 200,000 | 0.00% | 1.78 | 0.36 | 3.51 | 0.43 | 0.0% |
| Romanian | FLEURS 4,161, OpenSubtitles 200,000 | 0.00% | 2.10 | 0.46 | 4.03 | 0.47 | 0.0% |
| Swedish | FLEURS 3,474, OpenSubtitles 200,000 | 0.00% | 1.91 | 0.44 | 3.46 | 0.41 | 0.0% |
| Danish | FLEURS 3,790, OpenSubtitles 200,000 | 0.00% | 1.86 | 0.44 | 3.83 | 0.46 | 0.0% |
| Bulgarian | FLEURS 4,026, OpenSubtitles 200,000 | 0.00% | 2.60 | 0.56 | 5.01 | 0.55 | 0.0% |
| Russian | FLEURS 3,693, OpenSubtitles 200,000 | 0.00% | 2.23 | 0.44 | 3.83 | 0.44 | 0.0% |
| Czech | FLEURS 3,839, OpenSubtitles 200,000 | 0.00% | 2.62 | 0.55 | 4.79 | 0.52 | 0.0% |
| Greek | FLEURS 4,136, OpenSubtitles 200,000 | 0.00% | 4.89 | 0.97 | 11.27 | 1.07 | 63.8% |
| Ukrainian | FLEURS 3,885, OpenSubtitles 200,000 | 0.00% | 2.85 | 0.57 | 5.55 | 0.57 | 0.0% |
| Spanish | FLEURS 4,112, OpenSubtitles 200,000 | 0.00% | 1.78 | 0.38 | 3.48 | 0.45 | 0.0% |
| Dutch | FLEURS 3,453, OpenSubtitles 200,000 | 0.00% | 1.76 | 0.40 | 4.32 | 0.53 | 0.0% |
| Slovak | FLEURS 3,101, OpenSubtitles 200,000 | 0.00% | 2.52 | 0.53 | 4.73 | 0.51 | 0.0% |
| Croatian | FLEURS 4,752, OpenSubtitles 200,000 | 0.00% | 2.31 | 0.48 | 4.08 | 0.46 | 0.1% |
| Polish | FLEURS 3,937, OpenSubtitles 200,000 | 0.00% | 2.28 | 0.43 | 3.98 | 0.45 | 0.0% |
| Lithuanian | FLEURS 4,339, OpenSubtitles 200,000 | 0.00% | 2.86 | 0.51 | 4.53 | 0.48 | 0.0% |
| Norwegian Bokmal | FLEURS 3,687, OpenSubtitles 200,000 | 0.00% | 1.84 | 0.43 | 3.35 | 0.40 | 0.0% |
| Norwegian Nynorsk | Wikipedia 30,000 (not spoken) | 0.02% | 2.36 | 0.42 | - | - | - |
| Persian | FLEURS 4,341, OpenSubtitles 200,000 | 0.00% | 2.56 | 0.66 | 4.60 | 0.49 | 0.2% |
| Slovenian | FLEURS 3,695, OpenSubtitles 200,000 | 0.00% | 2.32 | 0.49 | 4.47 | 0.50 | 0.1% |
| Gujarati | FLEURS 4,577 | 0.00% | 5.06 | 1.00 | 9.91 | 0.95 | 35.7% |
| Latvian | FLEURS 3,317, OpenSubtitles 200,000 | 0.00% | 2.94 | 0.56 | 5.03 | 0.52 | 0.0% |
| Italian | FLEURS 4,286, OpenSubtitles 200,000 | 0.00% | 1.86 | 0.39 | 3.52 | 0.44 | 0.0% |
| Occitan | FLEURS 4,804 | 0.00% | 2.04 | 0.39 | 3.13 | 0.37 | 2.4% |
| Nepali | FLEURS 4,363 | 0.00% | 5.58 | 0.96 | 8.34 | 0.79 | 10.4% |
| Marathi | FLEURS 4,727 | 0.00% | 5.84 | 0.97 | 8.08 | 0.76 | 9.2% |
| Belarusian | FLEURS 3,808 | 0.00% | 3.69 | 0.61 | 5.24 | 0.53 | 0.1% |
| Serbian | FLEURS 3,934, OpenSubtitles 200,000 | 0.00% | 2.33 | 0.51 | 4.59 | 0.49 | 0.0% |
| Luxembourgish | FLEURS 3,844 | 0.00% | 2.46 | 0.42 | 4.40 | 0.50 | 0.0% |
| Venetian | Wikipedia 30,000 (not spoken) | 0.01% | 2.01 | 0.44 | - | - | - |
| Assamese | FLEURS 4,214 | 4.56% | 5.69 | 1.02 | 7.97 | 0.75 | 13.7% |
| Welsh | FLEURS 4,895 | 0.00% | 2.37 | 0.49 | 4.12 | 0.47 | 12.1% |
| Silesian | Wikipedia 30,000 (not spoken) | 0.01% | 2.96 | 0.50 | - | - | - |
| Asturian | FLEURS 3,855 | 0.00% | 1.94 | 0.37 | 3.94 | 0.48 | 0.0% |
| Chhattisgarhi | none | - | - | - | - | - | - |
| Awadhi | none | - | - | - | - | - | - |
| Maithili | Wikipedia 30,000 (not spoken) | 0.00% | 4.87 | 0.95 | - | - | - |
| Bhojpuri | Wikipedia 30,000 (not spoken) | 0.13% | 3.91 | 0.92 | - | - | - |
| Sindhi | FLEURS 4,849 | 0.00% | 2.75 | 0.73 | 5.32 | 0.58 | 0.5% |
| Irish | FLEURS 4,056 | 0.00% | 2.40 | 0.49 | 3.90 | 0.44 | 0.0% |
| Faroese | Wikipedia 30,000 (not spoken) | 0.00% | 2.70 | 0.50 | - | - | - |
| Hindi | FLEURS 2,777, OpenSubtitles 143,563 | 0.28% | 3.63 | 0.97 | 8.55 | 0.85 | 21.4% |
| Punjabi | FLEURS 2,748 | 3.17% | 4.25 | 1.02 | 8.57 | 0.85 | 26.0% |
| Bengali | FLEURS 4,328, OpenSubtitles 200,000 | 5.11% | 4.75 | 1.02 | 8.24 | 0.77 | 8.2% |
| Oriya | FLEURS 2,356 | 0.70% | 5.89 | 1.00 | 9.67 | 0.91 | 27.9% |
| Tajik | FLEURS 3,138 | 0.00% | 3.39 | 0.60 | 5.23 | 0.54 | 0.6% |
| Eastern Yiddish | Wikipedia 30,000 (not spoken) | 0.09% | 2.91 | 0.61 | - | - | - |
| Lombard | Wikipedia 30,000 (not spoken) | 0.01% | 2.09 | 0.47 | - | - | - |
| Ligurian | Wikipedia 30,000 (not spoken) | 0.11% | 2.35 | 0.48 | - | - | - |
| Sicilian | Wikipedia 30,000 (not spoken) | 0.00% | 2.38 | 0.45 | - | - | - |
| Friulian | Wikipedia 24,143 (not spoken) | 0.01% | 2.00 | 0.43 | - | - | - |
| Sardinian | Wikipedia 30,000 (not spoken) | 0.01% | 2.18 | 0.43 | - | - | - |
| Galician | FLEURS 3,497, OpenSubtitles 200,000 | 0.00% | 1.88 | 0.40 | 3.89 | 0.48 | 0.0% |
| Catalan | FLEURS 3,644, OpenSubtitles 200,000 | 0.00% | 1.96 | 0.43 | 3.78 | 0.46 | 0.0% |
| Icelandic | FLEURS 1,008, OpenSubtitles 200,000 | 0.00% | 2.56 | 0.56 | 5.00 | 0.55 | 0.3% |
| Tosk Albanian | OpenSubtitles 200,000 | 0.00% | 2.33 | 0.53 | - | - | - |
| Limburgish | Wikipedia 30,000 (not spoken) | 0.01% | 2.35 | 0.44 | - | - | - |
| Dari | none | - | - | - | - | - | - |
| Afrikaans | FLEURS 1,494, OpenSubtitles 61,311 | 0.00% | 1.81 | 0.43 | 4.35 | 0.52 | 13.1% |
| Macedonian | FLEURS 3,725, OpenSubtitles 200,000 | 0.00% | 2.63 | 0.56 | 5.13 | 0.56 | 0.0% |
| Sinhala | OpenSubtitles 200,000 | 0.00% | 4.76 | 0.98 | - | - | - |
| Urdu | FLEURS 2,675, OpenSubtitles 46,523 | 0.08% | 2.94 | 0.83 | 6.88 | 0.74 | 5.4% |
| Magahi | none | - | - | - | - | - | - |
| Bosnian | FLEURS 4,416, OpenSubtitles 200,000 | 0.00% | 2.27 | 0.49 | 4.09 | 0.46 | 0.0% |
| Armenian | FLEURS 4,380, OpenSubtitles 4,092 | 0.00% | 6.27 | 1.00 | 10.30 | 0.95 | 41.4% |
| Simplified Chinese | FLEURS 4,600, OpenSubtitles 200,000 | 0.01% | 5.18 | 0.70 | 2.44 | 0.21 | 0.0% |
| Traditional Chinese | OpenSubtitles 200,000 | 0.54% | 5.52 | 0.80 | - | - | - |
| Cantonese | FLEURS 3,120 | 0.03% | 16.77 | 0.78 | 2.27 | 0.19 | 0.0% |
| Burmese | FLEURS 4,322 | 0.00% | 14.44 | 0.99 | 9.93 | 0.85 | 24.2% |
| Standard Arabic | FLEURS 2,827, OpenSubtitles 200,000 | 0.00% | 2.11 | 0.49 | 3.64 | 0.43 | 0.1% |
| Najdi Arabic | none | - | - | - | - | - | - |
| Levantine Arabic | none | - | - | - | - | - | - |
| Egyptian Arabic | Wikipedia 30,000 (not spoken) | 0.07% | 2.20 | 0.49 | - | - | - |
| Moroccan Arabic | Wikipedia 30,000 (not spoken) | 0.04% | 2.52 | 0.55 | - | - | - |
| Mesopotamian Arabic | none | - | - | - | - | - | - |
| Taizzi-Adeni Arabic | none | - | - | - | - | - | - |
| Tunisian Arabic | none | - | - | - | - | - | - |
| Hebrew | FLEURS 4,362, OpenSubtitles 200,000 | 0.02% | 2.05 | 0.48 | 3.64 | 0.42 | 0.3% |
| Maltese | FLEURS 4,225 | 0.00% | 3.09 | 0.47 | 4.48 | 0.47 | 0.0% |
| Indonesian | FLEURS 3,616, OpenSubtitles 200,000 | 0.00% | 2.14 | 0.39 | 3.19 | 0.38 | 0.0% |
| Malay | FLEURS 3,740, OpenSubtitles 200,000 | 0.00% | 2.12 | 0.39 | 3.37 | 0.39 | 0.0% |
| Tagalog | FLEURS 3,266, OpenSubtitles 19,314 | 0.00% | 2.12 | 0.42 | 3.39 | 0.40 | 0.0% |
| Cebuano | FLEURS 4,027 | 0.00% | 1.95 | 0.38 | 3.55 | 0.43 | 0.1% |
| Javanese | FLEURS 4,074 | 0.00% | 2.37 | 0.40 | 3.39 | 0.39 | 0.0% |
| Sundanese | Wikipedia 30,000 (not spoken) | 0.05% | 2.21 | 0.37 | - | - | - |
| Minangkabau | Wikipedia 30,000 (not spoken) | 0.06% | 2.31 | 0.39 | - | - | - |
| Balinese | Wikipedia 30,000 (not spoken) | 1.11% | 2.49 | 0.42 | - | - | - |
| Banjar | Wikipedia 30,000 (not spoken) | 0.02% | 2.41 | 0.38 | - | - | - |
| Pangasinan | Wikipedia 13,534 (not spoken) | 0.01% | 1.82 | 0.36 | - | - | - |
| Iloko | Wikipedia 30,000 (not spoken) | 0.12% | 2.12 | 0.41 | - | - | - |
| Waray | Wikipedia 30,000 (not spoken) | 0.00% | 2.00 | 0.39 | - | - | - |
| Tamil | FLEURS 3,335, OpenSubtitles 40,165 | 0.00% | 6.95 | 0.99 | 10.01 | 0.90 | 36.2% |
| Telugu | FLEURS 3,085, OpenSubtitles 30,416 | 0.00% | 6.02 | 0.98 | 8.87 | 0.81 | 15.8% |
| Kannada | FLEURS 3,489 | 0.00% | 7.59 | 1.00 | 9.13 | 0.83 | 8.9% |
| Malayalam | FLEURS 4,419, OpenSubtitles 200,000 | 0.00% | 7.82 | 0.99 | 10.16 | 0.90 | 36.6% |
| Turkish | FLEURS 3,607, OpenSubtitles 200,000 | 0.00% | 2.44 | 0.41 | 3.34 | 0.38 | 0.0% |
| North Azerbaijani | FLEURS 3,988 | 0.00% | 3.48 | 0.53 | 5.10 | 0.53 | 0.0% |
| Northern Uzbek | FLEURS 4,168 | 0.00% | 3.14 | 0.43 | 4.47 | 0.47 | 0.0% |
| Kazakh | FLEURS 4,425, OpenSubtitles 4,100 | 0.00% | 4.43 | 0.67 | 5.39 | 0.53 | 0.0% |
| Bashkir | Wikipedia 30,000 (not spoken) | 0.01% | 4.38 | 0.68 | - | - | - |
| Tatar | Wikipedia 30,000 (not spoken) | 0.01% | 3.88 | 0.61 | - | - | - |
| Thai | FLEURS 4,062, OpenSubtitles 200,000 | 0.00% | 5.51 | 0.37 | 3.14 | 0.28 | 0.0% |
| Lao | FLEURS 2,405 | 0.00% | 19.80 | 0.96 | 7.93 | 0.67 | 8.1% |
| Finnish | FLEURS 4,037, OpenSubtitles 200,000 | 0.00% | 2.82 | 0.46 | 4.14 | 0.43 | 0.0% |
| Estonian | FLEURS 3,781, OpenSubtitles 200,000 | 0.00% | 2.44 | 0.47 | 4.25 | 0.46 | 0.0% |
| Hungarian | FLEURS 4,407, OpenSubtitles 200,000 | 0.00% | 2.69 | 0.49 | 4.89 | 0.52 | 0.0% |
| Vietnamese | FLEURS 4,212, OpenSubtitles 200,000 | 0.47% | 1.44 | 0.42 | 3.26 | 0.47 | 0.0% |
| Khmer | FLEURS 2,772 | 0.00% | 24.91 | 0.97 | 9.22 | 0.77 | 8.3% |
| Japanese | FLEURS 3,208, OpenSubtitles 200,000 | 0.01% | 5.25 | 0.70 | 2.87 | 0.24 | 0.0% |
| Korean | FLEURS 2,915, OpenSubtitles 200,000 | 0.01% | 2.38 | 0.81 | 3.06 | 0.34 | 0.0% |
| Georgian | FLEURS 2,879, OpenSubtitles 200,000 | 0.00% | 4.71 | 0.99 | 10.26 | 0.93 | 36.7% |
| Basque | OpenSubtitles 200,000 | 0.00% | 2.60 | 0.45 | - | - | - |
| Haitian | Wikipedia 30,000 (not spoken) | 0.00% | 2.18 | 0.48 | - | - | - |
| Papiamento | Wikipedia 20,427 (not spoken) | 0.00% | 1.92 | 0.42 | - | - | - |
| Kabuverdianu | FLEURS 3,945 | 0.00% | 2.01 | 0.42 | 3.15 | 0.38 | 0.0% |
| Tok Pisin | Wikipedia 3,215 (not spoken) | 0.18% | 2.03 | 0.43 | - | - | - |
| Swahili | FLEURS 3,768 | 0.00% | 2.44 | 0.44 | 3.20 | 0.36 | 0.0% |

[spoken measurement] 644 s since the start


### Word timings with Whisper (KsponSpeech and FLEURS)

Real word onsets are measured with Whisper large-v3 (S2.3): the reference transcript is teacher forced through the decoder and
dynamic time warping over the cross attention of its alignment heads gives each word its onset (`whisper_alignment`). The data
is KsponSpeech's evaluation split (6,000 utterances, spoken form of the transcript) and, for every language with FLEURS, the
first 200 utterances of its `dev` split. Languages Whisper lacks are prompted with a close one. The job runs on a Kaggle GPU
through letify: KsponSpeech travels through the Drive store, FLEURS is read from the hub on the remote machine, and the result
is saved as `word_timings.json` in the output folder, which later runs reuse. Delete that file to measure again.

In [10]:
import importlib.metadata
import zipfile

WORD_TIMINGS = OUT / "word_timings.json"
KSPONSPEECH = Path(".cache/datasets/ksponspeech")
WORD_TIMING_DEVICE = ("kaggle_60", "T4")
WORD_TIMING_UTTERANCES = 200
# Whisper's token where it differs from the FLEURS prefix, and a close language where Whisper has none.
WHISPER_LANGUAGE = {"cmn": "zh", "fil": "tl", "nb": "no", "jv": "jw", "ast": "es", "ga": "en", "or": "bn", "ceb": "tl", "kea": "pt"}
# Packages the remote machine installs beyond the lock, pinned to the versions here.
REMOTE_PACKAGES = ("sentencepiece", "protobuf", "pyloudnorm", "pyarrow", "librosa", "soundfile")


def whisper_language(fleurs_code):
    prefix = fleurs_code.split("_")[0]
    return WHISPER_LANGUAGE.get(prefix, prefix)


def package_zip():
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path("polyplex_research").rglob("*.py")):
            archive.write(path, str(path))
    return buffer.getvalue()


def whisper_aligner(model, processor, whisper_alignment):
    # Returns align(clips, language) for clips of (id, transcript, 16 kHz waveform), one row per clip kept.
    import torch

    def align_chunk(chunk, language):
        # The cross attention of every layer is kept, so a batch of long transcripts can exceed the card; it is
        # then split in half until it fits. The split runs after the except block, once the failed batch's
        # tensors are released.
        try:
            return whisper_alignment(model, processor, [c[2] for c in chunk], [c[1] for c in chunk], language=language)
        except torch.OutOfMemoryError:
            if len(chunk) == 1:
                raise
        torch.cuda.empty_cache()
        half = len(chunk) // 2
        return align_chunk(chunk[:half], language) + align_chunk(chunk[half:], language)

    def decoder_length(text):
        # Prompt (start, language, task, no timestamps), the word tokens and the end token, as whisper_alignment builds it.
        return 4 + sum(len(processor.tokenizer.encode(" " + word, add_special_tokens=False)) for word in text.split()) + 1

    def align(clips, language):
        # Whisper reads at most 30 seconds and its decoder at most max_target_positions tokens; a longer transcript
        # indexes past the position table and breaks the CUDA context. A clip without words has nothing to place.
        limit = model.config.max_target_positions
        kept = [clip for clip in clips
                if clip[1].split() and len(clip[2]) <= 30 * 16000 and decoder_length(clip[1]) <= limit]
        if len(kept) < len(clips):
            print(f"{language}: {len(clips) - len(kept)} of {len(clips)} clips left out (empty, over 30 s or over {limit} tokens)", flush=True)
        clips = kept
        rows = []
        for start in range(0, len(clips), 8):
            chunk = clips[start:start + 8]
            for (clip_id, text, waveform), (starts, ends, loss) in zip(chunk, align_chunk(chunk, language)):
                rows.append({"id": clip_id, "text": text, "seconds": len(waveform) / 16000,
                             "starts": [float(s) for s in starts], "ends": [float(e) for e in ends], "loss": float(loss)})
        return rows

    return align


def remote_word_timings(package, fleurs_codes, languages, per_language, ksponspeech):
    # Runs on the remote GPU, where the package arrives as a zip.
    import io
    import json
    import os
    import sys
    import tarfile
    import tempfile
    import zipfile

    import letify
    import numpy as np
    import soundfile
    import torch
    from huggingface_hub import hf_hub_download
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    root = tempfile.mkdtemp(prefix="package-")
    zipfile.ZipFile(io.BytesIO(package)).extractall(root)
    sys.path.insert(0, root)
    from polyplex_research.datasets.duplex_data import whisper_alignment
    from polyplex_research.datasets.ksponspeech import normalize_transcription

    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3", torch_dtype=torch.float16, attn_implementation="eager").to("cuda").eval()

    align = whisper_aligner(model, processor, whisper_alignment)

    def save(result):
        # On the Drive store after every source, for a connection lost on the way back. The call's own return
        # value carries the result, so a Drive that cannot be reached only costs this copy.
        path = os.path.join(tempfile.mkdtemp(prefix="word-timings-"), "word_timings.json")
        with open(path, "w", encoding="utf-8") as sink:
            json.dump(result, sink, ensure_ascii=False)
        try:
            letify.stash("word-timings-result", path)
        except Exception as error:
            print("Drive copy failed:", type(error).__name__, error, flush=True)

    def ksponspeech_rows():
        data = letify.unstash(ksponspeech, tempfile.mkdtemp(prefix="ksponspeech-"))
        clips = []
        for split in ("eval_clean", "eval_other"):
            for line in open(os.path.join(data, f"{split}.trn"), encoding="utf-8"):
                path, text = line.rstrip("\n").split(" :: ", 1)
                name = os.path.basename(path)
                samples = np.fromfile(os.path.join(data, split, name), dtype="<i2").astype(np.float32) / 32768
                clips.append((name.rsplit(".", 1)[0], normalize_transcription(text, "phonetic"), samples))
        return align(clips, "ko")

    def fleurs_rows(code):
        transcripts = hf_hub_download("google/fleurs", f"data/{code}/dev.tsv", repo_type="dataset")
        audio = hf_hub_download("google/fleurs", f"data/{code}/audio/dev.tar.gz", repo_type="dataset")
        wanted = {}
        for line in open(transcripts, encoding="utf-8"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 3 and fields[1] not in wanted and len(wanted) < per_language:
                wanted[fields[1]] = fields[2]
        waveforms = {}
        with tarfile.open(audio, "r:gz") as archive:
            for member in archive:
                name = os.path.basename(member.name)
                if member.isfile() and name in wanted:
                    samples, _ = soundfile.read(io.BytesIO(archive.extractfile(member).read()), dtype="float32")
                    waveforms[name] = np.asarray(samples, dtype=np.float32).reshape(-1)
        # One archive on disk at a time.
        os.remove(os.path.realpath(audio))
        clips = [(name.rsplit(".", 1)[0], text, waveforms[name]) for name, text in wanted.items() if name in waveforms]
        return align(clips, languages[code])

    # A source that fails is recorded and skipped, so the call returns everything that did finish.
    result, failed = {}, {}
    jobs = ([("ksponspeech", ksponspeech_rows)] if ksponspeech else []) + [(code, lambda code=code: fleurs_rows(code)) for code in fleurs_codes]
    for name, job in jobs:
        try:
            result[name] = job()
        except Exception as error:
            failed[name] = f"{type(error).__name__}: {error}"
            torch.cuda.empty_cache()
            print(name, "failed:", failed[name], flush=True)
            continue
        save(result)
        print(name, len(result[name]), flush=True)
    return {"timings": result, "failed": failed}


WORD_TIMINGS_PARTIAL = OUT / "word_timings_partial.json"

if not WORD_TIMINGS.exists():
    import letify

    # What earlier runs finished is kept in the partial file and not measured again.
    done = json.load(open(WORD_TIMINGS_PARTIAL)) if WORD_TIMINGS_PARTIAL.exists() else {}
    launcher = letify.Launcher()
    alias, accelerator = WORD_TIMING_DEVICE
    device = getattr(getattr(launcher.providers, alias), accelerator)
    env = letify.Env.from_lock().pip_install(*(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
    # One attempt per call: a retry would start every source again, and this cell already skips finished ones.
    run = launcher.function(device=device, host=letify.remote, env=env, retries=1)(remote_word_timings)
    if "ksponspeech" not in done:
        letify.stash("ksponspeech-eval", KSPONSPEECH)
    codes = [language.fleurs for language in QWEN3_LANGUAGES if language.fleurs and language.fleurs not in done]
    try:
        returned = run(package_zip(), codes, {code: whisper_language(code) for code in codes}, WORD_TIMING_UTTERANCES,
                       None if "ksponspeech" in done else "ksponspeech-eval")
    except Exception:
        # The call itself was lost; the Drive store holds every source it finished.
        partial = letify.unstash("word-timings-result", OUT / "word_timings_remote.json")
        if partial is not None:
            done.update(json.load(open(partial)))
            json.dump(done, open(WORD_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
        raise
    done.update(returned["timings"])
    json.dump(done, open(WORD_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
    if returned["failed"]:
        raise RuntimeError(f"Word timings failed for {returned['failed']}; run this cell again to measure only those.")
    json.dump(done, open(WORD_TIMINGS, "w"), ensure_ascii=False)
word_timings = json.load(open(WORD_TIMINGS))
print(f"Word timings: {len(word_timings.get('ksponspeech', [])):,} KsponSpeech utterances, "
      f"{sum(len(rows) for name, rows in word_timings.items() if name != 'ksponspeech'):,} FLEURS utterances in "
      f"{len(word_timings) - ('ksponspeech' in word_timings)} languages")

Word timings: 6,000 KsponSpeech utterances, 15,257 FLEURS utterances in 79 languages


### Word timings with Whisper on real speech (YODAS2)

FLEURS is read speech, and read speech leaves longer gaps between words than conversation, so it understates the cut rate
(S2.3). Real speech for every language comes from YODAS2 (`espnet/yodas2`, YouTube videos under CC BY, about 85 of the
Qwen3 languages), the user-uploaded subtitle subsets (`000`). For each language the subtitle JSON of the first shard is read,
and the audio shard is streamed from its start until enough videos are read, so the 1.5 GB shard is never downloaded whole.
Subtitle boundaries can be off, so the neighbouring subtitle segments of a video are joined into pieces of at most 25 seconds,
split at pauses over one second, with 0.3 seconds of margin on both sides; a boundary error then only touches the two ends of a
piece. Each piece is aligned with Whisper against its subtitle text, and the alignment loss is kept so pieces whose subtitles
do not match the speech can be left out. Korean was measured first and compared with KsponSpeech. Every language with a user-subtitle subset gets 1,000 pieces, 250
from each of four shards spread over the subset so one channel does not fill the sample. The languages are split into one group per
device and each group runs in its own process, so the devices work at the same time: the department server's card 0 (a Tesla
P100, which the CUDA 12.8 build of PyTorch does not support, so that environment is synced again with the `cu126` group) and
the T4s of two Kaggle accounts. Each device group runs in its own Python process that reads this notebook's cells, because the
department server's call did not start when one process drove several devices, from threads or from forked children, while
the same call alone went through.

In [11]:
YODAS_TIMINGS = OUT / "yodas_timings.json"
YODAS_TIMINGS_PARTIAL = OUT / "yodas_timings_partial.json"
# The CUDA 12.8 build of PyTorch has no kernels for the P100 (sm_60), so its environment is synced again with the cu126
# group after letify's default sync. The sync is exact so the cu128-only packages go, and the packages letify installed on
# top of the lock are installed again after it.
UV_ON_REMOTE = ("UV_PYTHON_INSTALL_DIR=/workspace/uv-python UV_CACHE_DIR=/workspace/uv-cache "
                "\"$(command -v uv || echo /workspace/uv-bin/uv)\"")
CU126_SYNC = (f"{UV_ON_REMOTE} sync --frozen --no-install-project --no-group cu128 --group cu126 --python 3.13",
              f"{UV_ON_REMOTE} pip install --python .venv/bin/python "
              + " ".join(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
# One group of languages per device: (provider, accelerator, extra environment commands).
YODAS_DEVICES = [("dept_gpu", "Tesla_P100", CU126_SYNC), ("kaggle_60", "T4", ()), ("kaggle_30", "T4", ())]
YODAS_PIECES = 1000
YODAS_SHARDS = 4
# YODAS2 code where it is not the Wikipedia code of the table, and the Whisper token where Whisper spells the code
# differently or lacks the language (a close language prompts the forced alignment).
YODAS_CODE = {"Hebrew": "iw", "Bhojpuri": "bh", "Simplified Chinese": "zh", "Norwegian Bokmal": "no"}
YODAS_WHISPER = {"iw": "he", "jv": "jw", "bh": "hi", "ga": "en", "or": "bn", "sc": "it"}


def yodas_subsets():
    from huggingface_hub import HfApi

    folders = {f.path.split("/")[1] for f in HfApi().list_repo_tree("espnet/yodas2", path_in_repo="data", repo_type="dataset")}
    subsets = {}
    for language in QWEN3_LANGUAGES:
        code = YODAS_CODE.get(language.name, language.wikipedia)
        # The 000 subsets carry the subtitles users uploaded.
        if code and f"{code}000" in folders and f"{code}000" not in subsets:
            subsets[f"{code}000"] = YODAS_WHISPER.get(code, code)
    return subsets


YODAS_SUBSETS = yodas_subsets()


def remote_yodas_timings(package, subsets, pieces_per_language, shards, stash_name):
    # Runs on the remote GPU, where the package arrives as a zip.
    import io
    import json
    import os
    import sys
    import tarfile
    import tempfile
    import zipfile

    import letify
    import librosa
    import numpy as np
    import soundfile
    import torch
    from huggingface_hub import HfFileSystem, hf_hub_download
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    root = tempfile.mkdtemp(prefix="package-")
    zipfile.ZipFile(io.BytesIO(package)).extractall(root)
    sys.path.insert(0, root)
    from polyplex_research.datasets.duplex_data import whisper_alignment

    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3", torch_dtype=torch.float16, attn_implementation="eager").to("cuda").eval()
    align = whisper_aligner(model, processor, whisper_alignment)
    fs = HfFileSystem()

    def pieces_of(audio_id, segments, waveform):
        # Subtitle ids end in -<index>-<start>-<end>, the times in hundredths of a second.
        timed = []
        for utt_id, text in segments.items():
            _, _, start, end = utt_id.rsplit("-", 3)
            if text.split():
                timed.append((int(start) / 100, int(end) / 100, " ".join(text.split())))
        timed.sort()
        pieces, current = [], []

        def flush():
            if current:
                begin = max(0.0, current[0][0] - 0.3)
                finish = min(len(waveform) / 16000, current[-1][1] + 0.3)
                clip = waveform[int(begin * 16000):int(finish * 16000)]
                pieces.append((f"{audio_id}-{begin:.2f}", " ".join(t for _, _, t in current), clip))

        for segment in timed:
            if current and (segment[1] - current[0][0] > 25 or segment[0] - current[-1][1] > 1.0):
                flush()
                current = []
            current.append(segment)
        flush()
        return pieces

    def shard_rows(subset, shard, language, wanted):
        videos = json.load(open(hf_hub_download("espnet/yodas2", f"data/{subset}/text/{shard}.json", repo_type="dataset")))
        segments = {video["audio_id"]: video["text"] for video in videos}
        rows = []
        # Streamed from the start and closed once enough pieces are aligned, so only a prefix of the shard is read.
        with fs.open(f"datasets/espnet/yodas2/data/{subset}/audio/{shard}.tar.gz", "rb", block_size=4 * 2**20) as raw:
            with tarfile.open(fileobj=raw, mode="r|gz") as archive:
                for member in archive:
                    audio_id = os.path.basename(member.name).rsplit(".", 1)[0]
                    if not member.isfile() or audio_id not in segments:
                        continue
                    samples, rate = soundfile.read(io.BytesIO(archive.extractfile(member).read()), dtype="float32")
                    if samples.ndim > 1:
                        samples = samples.mean(axis=1)
                    waveform = librosa.resample(samples, orig_sr=rate, target_sr=16000) if rate != 16000 else samples
                    rows += align(pieces_of(audio_id, segments[audio_id], waveform), language)
                    if len(rows) >= wanted:
                        break
        return rows[:wanted]

    def subset_rows(subset, language):
        names = sorted(os.path.basename(f["name"]).rsplit(".", 1)[0]
                       for f in fs.ls(f"datasets/espnet/yodas2/data/{subset}/text", detail=True))
        # Shards spread evenly over the subset, each giving an equal share of the pieces.
        picked = [names[round(k * (len(names) - 1) / max(shards - 1, 1))] for k in range(min(shards, len(names)))]
        picked = list(dict.fromkeys(picked))
        rows = []
        for k, shard in enumerate(picked):
            wanted = (pieces_per_language - len(rows)) // (len(picked) - k)
            rows += shard_rows(subset, shard, language, wanted)
        return rows

    def save(result):
        path = os.path.join(tempfile.mkdtemp(prefix="yodas-timings-"), "yodas_timings.json")
        with open(path, "w", encoding="utf-8") as sink:
            json.dump(result, sink, ensure_ascii=False)
        try:
            letify.stash(stash_name, path)
        except Exception as error:
            print("Drive copy failed:", type(error).__name__, error, flush=True)

    # A subset that fails is recorded and skipped, so the call returns everything that did finish.
    result, failed = {}, {}
    for subset, language in subsets.items():
        try:
            result[subset] = subset_rows(subset, language)
        except Exception as error:
            failed[subset] = f"{type(error).__name__}: {error}"
            torch.cuda.empty_cache()
            print(subset, "failed:", failed[subset], flush=True)
            continue
        save(result)
        print(subset, len(result[subset]), flush=True)
    return {"timings": result, "failed": failed}


def complete(timings):
    # A subset with a result is done; one with fewer than YODAS_PIECES pieces ran out of videos in its shards, which a
    # second run would not change.
    return {subset: rows for subset, rows in timings.items() if rows}


def run_group(k, group):
    # Runs in its own forked process with its own letify launcher; returns what the group finished and what failed.
    import letify

    alias, accelerator, commands = YODAS_DEVICES[k]
    env = letify.Env.from_lock().pip_install(*(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
    if commands:
        env = env.run(*commands)
    launcher = letify.Launcher()
    device = getattr(getattr(launcher.providers, alias), accelerator)
    run = launcher.function(device=device, host=letify.remote, env=env, retries=1)(remote_yodas_timings)
    stash_name = f"yodas-timings-{alias}"
    try:
        returned = run(package_zip(), group, YODAS_PIECES, YODAS_SHARDS, stash_name)
        return returned["timings"], returned["failed"]
    except Exception as error:
        partial = letify.unstash(stash_name, OUT / f"yodas_timings_remote_{alias}.json")
        timings = json.load(open(partial)) if partial is not None else {}
        return timings, {subset: f"{type(error).__name__}: {error}" for subset in group if subset not in timings}


# A child interpreter runs one device group: it executes the notebook cells that define the language table, the Whisper
# aligner and this cell's functions, then writes the group's result to a file.
GROUP_RUNNER = """
import io, json, sys, nbformat, __main__
cells = nbformat.read("build_vocab.ipynb", 4).cells
names = __main__.__dict__
names["io"] = io
for marker in ("QWEN3_LANGUAGES = (", "def whisper_aligner(", "def run_group("):
    source = next(cell.source for cell in cells if cell.cell_type == "code" and marker in cell.source)
    exec(source[:source.index("\\nif not YODAS_TIMINGS.exists():")] if marker == "def run_group(" else source, names)
timings, failed = names["run_group"](int(sys.argv[1]), json.loads(sys.argv[2]))
json.dump({"timings": timings, "failed": failed}, open(sys.argv[3], "w"), ensure_ascii=False)
"""


if not YODAS_TIMINGS.exists():
    import subprocess
    import sys
    import time

    done = complete(json.load(open(YODAS_TIMINGS_PARTIAL))) if YODAS_TIMINGS_PARTIAL.exists() else {}
    todo = [subset for subset in YODAS_SUBSETS if subset not in done]
    groups = [{subset: YODAS_SUBSETS[subset] for subset in todo[k::len(YODAS_DEVICES)]} for k in range(len(YODAS_DEVICES))]
    children = {}
    for k, group in enumerate(groups):
        if group:
            alias = YODAS_DEVICES[k][0]
            result = OUT / f"yodas_timings_{alias}.json"
            result.unlink(missing_ok=True)
            log = open(OUT / f"yodas_timings_{alias}.log", "w")
            children[k] = (subprocess.Popen([sys.executable, "-c", GROUP_RUNNER, str(k), json.dumps(group), str(result)],
                                            stdout=log, stderr=subprocess.STDOUT), result)
    failed = {}
    # Each group's result is merged and saved as soon as its process ends.
    while children:
        for k, (child, result) in list(children.items()):
            if child.poll() is None:
                continue
            returned = json.load(open(result)) if result.exists() else {"timings": {}, "failed": {s: f"process exited {child.returncode}" for s in groups[k]}}
            done.update(returned["timings"])
            failed.update(returned["failed"])
            json.dump(done, open(YODAS_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
            del children[k]
        time.sleep(30)
    if failed:
        raise RuntimeError(f"YODAS2 timings failed for {failed}; run this cell again to measure only those.")
    json.dump(done, open(YODAS_TIMINGS, "w"), ensure_ascii=False)
yodas_timings = json.load(open(YODAS_TIMINGS))
print({subset: len(rows) for subset, rows in yodas_timings.items()})

{'pt000': 972, 'sv000': 1000, 'ru000': 1000, 'uk000': 913, 'sk000': 1000, 'lt000': 1000, 'sl000': 1000, 'it000': 1000, 'mr000': 575, 'lb000': 9, 'bh000': 1, 'fo000': 10, 'bn000': 472, 'yi000': 8, 'ca000': 1000, 'af000': 72, 'ur000': 638, 'zh000': 860, 'iw000': 1000, 'jv000': 202, 'te000': 698, 'tr000': 1000, 'kk000': 458, 'th000': 883, 'et000': 1000, 'km000': 444, 'ka000': 1000, 'sw000': 633, 'en000': 1000, 'de000': 785, 'da000': 1000, 'cs000': 1000, 'es000': 766, 'hr000': 1000, 'no000': 838, 'gu000': 465, 'oc000': 28, 'be000': 1000, 'as000': 104, 'sd000': 13, 'hi000': 1000, 'or000': 297, 'sc000': 2, 'is000': 654, 'mk000': 787, 'bs000': 801, 'my000': 299, 'id000': 1000, 'su000': 13, 'kn000': 591, 'az000': 799, 'ba000': 14, 'lo000': 25, 'hu000': 1000, 'ja000': 770, 'eu000': 1000, 'fr000': 1000, 'ro000': 1000, 'bg000': 753, 'el000': 1000, 'nl000': 1000, 'pl000': 994, 'fa000': 797, 'lv000': 328, 'ne000': 556, 'sr000': 1000, 'cy000': 487, 'ga000': 182, 'pa000': 467, 'tg000': 47, 'gl000': 5

## 5. Tiers 3 and 2: subwords first, then frequent word forms

Each language's spoken data from section 4 is split into 80% training and 20% evaluation (S2.3), with at most 20,000
evaluation lines per language. Korean also uses the v1 spoken corpus (`korean_texts` from section 2).

- Scores: before selecting, Unigram EM is rerun on the training parts of every language mixed together (at most 30,000 lines
  per language), over the current vocabulary (Qwen3 tokens and tier 1). The section 3 scores were fitted on Korean and English
  only, so every other language's tokens sat at the floor score and its words were cut too finely.
- Word units: whitespace units for spaced languages. Scripts written without spaces (Han, kana, Thai, Lao, Khmer, Myanmar) are
  cut by a SentencePiece Unigram model trained on that language's training part (up to 8,000 pieces of at most 8 characters),
  and its pieces are the words. Trailing punctuation is removed from every word, because `text_stream` splits it off and places
  it only on free frames.
- Order (S2.3): each language runs both orders. (a) Tier 3 subwords first, then tier 2 whole words for what is still cut.
  (b) Tier 2 whole words first, then subwords for what is left. The order that reaches the target with fewer tokens is kept
  (the lower cut rate when neither reaches it). A subword shortens every word that holds it, a whole word only itself, so (a)
  wins where words take many tokens and (b) where a few hundred words are enough. Subwords come from a SentencePiece Unigram
  model trained on the language's multi-token training words; they are added 1,000 at a time while the cut rate falls,
  stopping at the target or after four batches without gain, up to 20,000. Languages already below their target get none.
- Token counts for both searches come from the Unigram EM of section 3 run on the language's own spoken words over the
  vocabulary in hand (tier 1, then tier 1 and the subwords). The first column of the table uses the mixed EM above instead.
- Candidates: words seen at least twice in the training part, together with the subtitles of up to 20 YODAS2 shards the
  evaluation did not read that take two or more tokens after the subwords, ordered by frames
  saved (count × (tokens − 1)).
- Evaluation: the cut rate is measured on real word onsets (S2.3), the YODAS2 pieces with an alignment loss up to the
  95th percentile on KsponSpeech, for every language including Korean. A piece where more than 10% of neighbouring
  word onsets are under 20 ms apart is also left out: Whisper did not align it (Bengali pieces put most words on one
  instant). Languages left with fewer than 2,000
  words, and scripts written without spaces, are measured on their evaluation lines with the approximation below.
- On real onsets a word whose span or gap to the next onset is under 30 ms per character is an alignment error. It keeps
  its frames but is counted neither as cut nor in the total (S2.2); tier 2 cannot change it.
- Candidates are added 250 at a time while the word cut rate (S2.2) is measured on the evaluation part. Stop when it falls below 1% or within 0.2 points of the lowest cut rate (every word one token); the main languages
  of the INTENT goal (English, Korean, Japanese, Chinese) go on to the latter even below 1%,
  when four batches in a row (1,000 forms) lower it by no more than 0.01 points each, keeping the forms up to the last batch that helped, or at 20,000 forms. Added forms and strings already in
  the vocabulary count as one token.
- Word cut approximation: word timings are missing, so a word's onset is the utterance duration times the share of characters
  before it. Words are then placed as `text_stream` places them (S1.3): the onset frame is `int(onset × 12.5)` and the tokens
  run from it. A word starting before the previous word's tokens end overwrites and cuts that word, and a word running past the
  end of the utterance is not written; both count as cut. FLEURS utterances use the recorded duration. Other lines get a duration from that
  language's FLEURS characters per second (the median over languages of the same script when the language has no FLEURS).

In [12]:
NO_SPACE_SCRIPTS = {"han", "kana", "thai", "lao", "khmer", "myanmar"}
FRAMES_PER_SECOND = 12.5
EVAL_LINES = 20_000
EM_LINES = 30_000
vocabulary = set(unigram_pieces(tier1_all))


def language_scripts(language):
    letters = [ch for ch in standard.get(language.name, set()) if len(ch) == 1 and ch.isalpha()]
    counts = collections.Counter(script_of(ch) for ch in letters)
    return {script for script, count in counts.items() if script and count >= 5}


def main_script(language):
    if language.name == "Japanese":
        return "kana"
    letters = [ch for ch in standard.get(language.name, set()) if len(ch) == 1 and ch.isalpha()]
    counts = collections.Counter(script_of(ch) for ch in letters)
    counts.pop(None, None)
    return counts.most_common(1)[0][0] if counts else None


def spoken_part(word):
    end = len(word)
    while end and unicodedata.category(word[end - 1])[0] == "P":
        end -= 1
    return word[:end]


IN_WORD_MARKS = {"'", "’", "-", "\u200c", "\u200d"}


def clean_form(word):
    # A tier 2 or tier 3 form is letters, combining marks and the symbols of a script (Sindhi `۾` is a word), with
    # apostrophes, hyphens, ZWJ and ZWNJ only inside it. Leading punctuation, digits, shared symbols such as `♪` and other
    # punctuation come from subtitle formatting, not from speech.
    if not word or word[0] in IN_WORD_MARKS or word[-1] in IN_WORD_MARKS:
        return False
    return all(unicodedata.category(ch)[0] in "LM" or ch in IN_WORD_MARKS
               or (unicodedata.category(ch)[0] == "S" and script_of(ch)) for ch in word)


def characters_per_second(durations):
    characters = sum(len(text.replace(" ", "")) for text in durations)
    return characters / sum(durations.values())


def unit_model(sentences):
    model = io.BytesIO()
    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(sentences), model_writer=model, model_type="unigram", vocab_size=8000, hard_vocab_limit=False,
        character_coverage=1.0, max_sentencepiece_length=8, add_dummy_prefix=False, minloglevel=2,
    )
    return spm.SentencePieceProcessor(model_proto=model.getvalue())


def unit_cutter(script, train_texts):
    if script not in NO_SPACE_SCRIPTS:
        return str.split
    model = unit_model(train_texts)
    return lambda text: [piece.replace("▁", "") for piece in model.encode(text, out_type=str) if piece.replace("▁", "")]


token_cache = {}


def token_count(word, added=()):
    # The stream cuts a word with `▁` in front, so a word is one token when its word-start form is.
    if "▁" + word in vocabulary or word in added:
        return 1
    if word not in token_cache:
        token_cache[word] = len(unigram_spoken.encode(word))
    return token_cache[word]


def count_tokens(words):
    # Fills the token cache for many words at once.
    missing = [word for word in set(words) if word not in token_cache and "▁" + word not in vocabulary]
    for word, ids in zip(missing, unigram_spoken.encode(missing, num_threads=THREADS)):
        token_cache[word] = len(ids)


def layout(units, seconds):
    # Onsets follow the share of characters before each unit; trailing punctuation still takes time but no frame.
    total = sum(len(unit) for unit in units)
    words, onsets, before = [], [], 0
    for unit in units:
        word = spoken_part(unit)
        if word:
            words.append(word)
            onsets.append(1 + int(seconds * before / total * FRAMES_PER_SECOND))
        before += len(unit)
    return words, onsets, 1 + math.ceil(seconds * FRAMES_PER_SECOND), [True] * len(words)


def cut_rate(layouts, added=(), counter=None):
    # Placed as text_stream places words (S1.3): a word starting before the previous word's tokens end cuts that word,
    # and a word running past the end of the utterance is not written. Both count as cut.
    cut = total = 0
    # A word whose timing is not plausible still takes its frames but is counted neither as cut nor in the total.
    for words, onsets, frames, plausible in layouts:
        previous_end = previous_plausible = None
        for word, onset, ok in zip(words, onsets, plausible):
            total += ok
            end = onset + (counter(word) if counter else token_count(word, added))
            if end > frames:
                cut += ok
                continue
            if previous_end is not None and onset < previous_end:
                cut += previous_plausible
            previous_end, previous_plausible = end, ok
    return cut / max(total, 1)


def placements(layouts, added=()):
    # [word, tokens, cut, written without EPAD, plausible] for every word, placed as cut_rate places it; cut is only
    # set on plausible words.
    rows = []
    for words, onsets, frames, plausible in layouts:
        previous_end = previous_row = None
        for word, onset, ok in zip(words, onsets, plausible):
            tokens = token_count(word, added)
            if onset + tokens > frames:
                rows.append([word, tokens, ok, False, ok])
                continue
            # The EPAD frame, onset - 1, still holds the previous word's last token.
            bare = previous_end is not None and previous_end >= onset
            if previous_end is not None and onset < previous_end:
                rows[previous_row][2] = rows[previous_row][4]
            rows.append([word, tokens, False, bare, ok])
            previous_end, previous_row = onset + tokens, len(rows) - 1
    return rows


scripts = {language.name: main_script(language) for language in QWEN3_LANGUAGES}
speed = {name: characters_per_second(data["durations"]) for name, data in spoken.items() if data["durations"]}
speed_by_script = collections.defaultdict(list)
for name, value in speed.items():
    speed_by_script[scripts[name]].append(value)
all_speeds = sorted(speed.values())

splits, cutters = {}, {}
for language in QWEN3_LANGUAGES:
    texts = [unicodedata.normalize("NFC", text) for text in
             list(spoken[language.name]["texts"]) + (korean_texts if language.name == "Korean" else [])]
    if not texts:
        continue
    order = list(range(len(texts)))
    random.Random(0).shuffle(order)
    cut = int(len(order) * 0.8)
    splits[language.name] = ([texts[i] for i in order[:cut]], [texts[i] for i in order[cut:]][:EVAL_LINES])
    cutters[language.name] = unit_cutter(scripts[language.name], splits[language.name][0])

# The EM sees the same word units that the cut rate places, so unspaced scripts enter as Unigram units, not whole lines.
em_words = [word for name, (train_texts, _) in splits.items() for text in train_texts[:EM_LINES] for word in cutters[name](text)]
print(f"Unigram EM on the training parts of {len(splits)} languages, {len(em_words):,} words")
unigram_spoken = fit_unigram(sorted(vocabulary), em_words)
(OUT / "unigram_spoken.model").write_bytes(unigram_spoken.serialized_model_proto())

import numpy as np

# Pieces whose alignment loss exceeds the 95th percentile on KsponSpeech are taken as subtitles that do not match the speech.
LOSS_LIMIT = float(np.percentile([row["loss"] for row in word_timings["ksponspeech"]], 95))
MIN_REAL_WORDS = 2000
# The languages of the INTENT goal (S2.2).
MAIN_LANGUAGES = {"English", "Korean", "Japanese", "Simplified Chinese", "Traditional Chinese"}
# A piece whose word onsets are often under 20 ms apart was not aligned; normal languages stay at 0.1 to 3%.
MAX_SQUEEZED = 0.10


def squeezed_share(row):
    gaps = np.diff(row["starts"])
    return float(np.mean(gaps < 0.02)) if len(gaps) else 0.0


# Under 30 ms per character a word cannot be spoken: such a span or onset gap is an alignment error.
MIN_SECONDS_PER_CHARACTER = 0.03


def real_layout(text, starts, seconds, ends):
    text = unicodedata.normalize("NFC", text)
    kept = [(spoken_part(unit), start, end) for unit, start, end in zip(text.split(), starts, ends) if spoken_part(unit)]
    words = [word for word, _, _ in kept]
    onsets = [1 + int(start * FRAMES_PER_SECOND) for _, start, _ in kept]
    plausible = []
    for k, (word, start, end) in enumerate(kept):
        gap = kept[k + 1][1] - start if k + 1 < len(kept) else None
        plausible.append((end - start) / len(word) >= MIN_SECONDS_PER_CHARACTER
                         and (gap is None or gap / len(word) >= MIN_SECONDS_PER_CHARACTER))
    return words, onsets, 1 + math.ceil(seconds * FRAMES_PER_SECOND), plausible


def real_layouts(language):
    # Real word onsets for the evaluation part: the YODAS2 pieces whose subtitles match the speech, the same source for
    # every language. KsponSpeech only checks this source on Korean.
    subset = f"{YODAS_CODE.get(language.name, language.wikipedia)}000"
    rows = [row for row in yodas_timings.get(subset, []) if row["loss"] <= LOSS_LIMIT and squeezed_share(row) <= MAX_SQUEEZED]
    source = "YODAS2 Whisper onsets"
    layouts = [real_layout(row["text"], row["starts"], row["seconds"], row["ends"]) for row in rows]
    if sum(len(words) for words, *_ in layouts) < MIN_REAL_WORDS:
        return None, None
    return layouts, source


YODAS_CANDIDATE_SHARDS = 20


def yodas_subtitle_lines(language):
    # Subtitles of the YODAS2 shards the evaluation did not read, as more spoken text for the tier 2 candidates.
    from huggingface_hub import HfApi, hf_hub_download

    subset = f"{YODAS_CODE.get(language.name, language.wikipedia)}000"
    if subset not in yodas_timings:
        return []
    names = sorted(f.path.rsplit("/", 1)[1].rsplit(".", 1)[0]
                   for f in HfApi().list_repo_tree("espnet/yodas2", path_in_repo=f"data/{subset}/text", repo_type="dataset"))
    evaluated = {names[round(k * (len(names) - 1) / max(YODAS_SHARDS - 1, 1))] for k in range(min(YODAS_SHARDS, len(names)))}
    lines = []
    for shard in [name for name in names if name not in evaluated][:YODAS_CANDIDATE_SHARDS]:
        videos = json.load(open(hf_hub_download("espnet/yodas2", f"data/{subset}/text/{shard}.json", repo_type="dataset")))
        lines += [unicodedata.normalize("NFC", " ".join(text.split())) for video in videos for text in video["text"].values()
                  if text.split()]
    return lines


REPLACEMENT_CHARACTERS = {"\ufffd", "\ufffc"}


def subword_pieces(words, budget):
    alphabet = {ch for word in words for ch in word}
    model = io.BytesIO()
    spm.SentencePieceTrainer.train(
        sentence_iterator=iter(words), model_writer=model, model_type="unigram",
        vocab_size=len(alphabet) + budget * 2 + 10, hard_vocab_limit=False, character_coverage=1.0,
        add_dummy_prefix=True, split_by_whitespace=True, minloglevel=2,
    )
    processor = spm.SentencePieceProcessor(model_proto=model.getvalue())
    pieces = [(processor.get_score(i), processor.id_to_piece(i)) for i in range(processor.get_piece_size())
              if not processor.is_unused(i) and not processor.is_control(i) and not processor.is_unknown(i)]
    return [piece for _, piece in sorted(pieces, reverse=True)]


def broken(ch):
    return unicodedata.category(ch) in ("Cc", "Co", "Cs", "Cf") or ch in REPLACEMENT_CHARACTERS or ch.isspace()


TIER3_BATCH = 1000
TIER3_MAX = 20_000


def spoken_counter(pieces, word_counts, words):
    # Token counts of `words` after the EM of section 3 on the language's spoken words. A piece with a character the
    # language never uses cannot cut its words, so only the other pieces enter the EM.
    alphabet = {ch for word in list(word_counts) + words for ch in word} | {"▁"}
    model = fit_unigram(sorted(piece for piece in pieces if set(piece) <= alphabet), word_counts, verbose=False)
    return dict(zip(words, map(len, model.encode(words, num_threads=THREADS))))


def sized_subwords(ordered, known, word_counts, layouts, target, start):
    # Adds subwords 1,000 at a time while the evaluation cut rate falls, as tier 2 adds word forms.
    words = sorted({word for layout_words, *_ in layouts for word in layout_words})
    kept, pending, stall, rate = [], [], 0, start
    for first in range(0, min(len(ordered), TIER3_MAX), TIER3_BATCH):
        if rate < target:
            break
        batch = ordered[first:first + TIER3_BATCH]
        counts = spoken_counter(known | set(kept) | set(pending) | set(batch), word_counts, words)
        trial = cut_rate(layouts, counter=counts.__getitem__)
        if rate - trial > 0.0001:
            kept, pending, stall, rate = kept + pending + batch, [], 0, trial
            continue
        pending += batch
        stall += 1
        if stall == 4:
            break
    return kept, rate



def search_tiers(language):
    global THREADS
    THREADS = WORKER_THREADS
    script = scripts[language.name]
    if language.name not in splits:
        return {"name": language.name, "tier2": [],
                "row": [language.name, script, "-", "-", "-", "-", "-", "-", "-", 0, 0, "-", "-", "no data"]}
    train_texts, eval_texts = splits[language.name]
    cutter = cutters[language.name]
    rate = speed.get(language.name)
    if rate is None:
        same = sorted(speed_by_script.get(script, [])) or all_speeds
        rate = same[len(same) // 2]
    layouts = []
    for text in eval_texts:
        units = cutter(text)
        if units:
            seconds = spoken[language.name]["durations"].get(text) or len(text.replace(" ", "")) / rate
            layouts.append(layout(units, seconds))
    timing = "approximation, FLEURS recording" if language.name in speed else "approximation, median characters per second"
    real, source = (None, None) if script in NO_SPACE_SCRIPTS else real_layouts(language)
    if real:
        layouts, timing = real, source
    candidate_texts = train_texts + (yodas_subtitle_lines(language) if script not in NO_SPACE_SCRIPTS else [])
    counts = collections.Counter(word for text in candidate_texts for word in map(spoken_part, cutter(text)) if word)
    count_tokens(list(counts) + [word for words, *_ in layouts for word in words])
    before = cut_rate(layouts)
    # The search stops at 1%, or once what one-token words could still save is under 0.2 points; the main languages go
    # on to that point even below 1%.
    lowest = cut_rate(layouts, {word for words, *_ in layouts for word in words})
    target = lowest + 0.002 if language.name in MAIN_LANGUAGES else max(0.01, lowest + 0.002)
    eval_words = sorted({word for words, *_ in layouts for word in words})
    every_word = sorted(set(counts) | set(eval_words))
    tokens = spoken_counter(vocabulary, counts, every_word)
    start = cut_rate(layouts, counter=tokens.__getitem__)

    def add_subwords(known, begin, exclude=()):
        if begin < target:
            return [], begin
        residual = [word for word, count in counts.items() if word not in exclude and token_count(word) >= 2
                    and clean_form(word) for _ in range(min(count, 50))]
        ordered = [piece for piece in subword_pieces(residual, TIER3_MAX)
                   if len(piece.lstrip("▁")) >= 2 and piece not in known and not any(broken(ch) for ch in piece)]
        return sized_subwords(ordered, known, counts, layouts, target, begin)

    def add_words(counted, begin):
        # Whole words on what is left, counted with the same EM; an added form is one token. A batch that does not help is
        # held back; four in a row end the search at the last batch that did.
        candidates = [(count * (counted[word] - 1), word) for word, count in counts.items()
                      if count >= 2 and counted[word] >= 2 and clean_form(word)]
        candidates = [word for _, word in sorted(candidates, reverse=True)]
        added, pending, stall, next_batch, rate = set(), set(), 0, 0, begin
        while rate >= target and len(added) + len(pending) < 20_000 and next_batch < len(candidates):
            batch = set(candidates[next_batch:next_batch + 250])
            next_batch += 250
            trial = cut_rate(layouts, counter=lambda word, chosen=added | pending | batch: 1 if word in chosen else counted[word])
            if rate - trial > 0.0001:
                added, pending, stall, rate = added | pending | batch, set(), 0, trial
                continue
            pending |= batch
            stall += 1
            if stall == 4:
                break
        return [word for word in candidates if word in added], rate

    # (a) Subwords first: a subword shortens every word that holds it.
    subwords_a, rate = add_subwords(vocabulary, start)
    counted = spoken_counter(vocabulary | set(subwords_a), counts, every_word) if subwords_a else tokens
    words_a, rate_a = add_words(counted, cut_rate(layouts, counter=counted.__getitem__))
    # (b) Whole words first, then subwords for what is left.
    words_b, rate = add_words(tokens, start)
    known = vocabulary | {"▁" + word for word in words_b}
    if rate >= target:
        rate = cut_rate(layouts, counter=spoken_counter(known, counts, eval_words).__getitem__)
    subwords_b, rate_b = add_subwords(known, rate, exclude=set(words_b))
    # Below the target with fewer tokens wins; when neither reaches it, the lower cut rate.
    options = {"subwords first": (words_a, subwords_a, rate_a), "words first": (words_b, subwords_b, rate_b)}
    order = min(options, key=lambda name: (options[name][2] >= target,
                                           len(options[name][0]) + len(options[name][1]) if options[name][2] < target
                                           else options[name][2]))
    words, subwords, current = options[order]
    return {"name": language.name, "tier2": words, "subwords": subwords, "counts": counts, "state": (layouts, target, current),
            "row": [language.name, script, "Unigram units" if script in NO_SPACE_SCRIPTS else "whitespace",
                    f"{len(candidate_texts):,}/{len(layouts):,}", f"{before:.2%}", f"{start:.2%}",
                    f"{len(subwords_a):,} + {len(words_a):,} → {rate_a:.2%}", f"{len(words_b):,} + {len(subwords_b):,} → {rate_b:.2%}",
                    order, len(subwords), len(words), f"{current:.2%}", f"{target:.2%}", timing]}


tier2, subwords_of, tier2_rows, unit_counts = {}, {}, [], {}
# What the final check needs from each language's search: the evaluation layouts, the target and the cut rate reached.
tier2_state = {}
for result in in_workers(search_tiers, QWEN3_LANGUAGES):
    tier2[result["name"]] = result["tier2"]
    subwords_of[result["name"]] = result.get("subwords", [])
    tier2_rows.append(result["row"])
    if "counts" in result:
        unit_counts[result["name"]] = result["counts"]
        tier2_state[result["name"]] = result["state"]

table(tier2_rows, ["Language", "Script", "Word unit", "Candidate lines/eval pieces", "Cut rate (mixed EM, tier 1)",
                   "Cut rate (language EM, tier 1)", "(a) subwords + words", "(b) words + subwords", "Order kept",
                   "Tier 3 subwords", "Tier 2", "Cut rate after both", "Target", "Word timing"])
print(f"Tier 3 subwords summed over languages: {sum(len(v) for v in subwords_of.values()):,}; tier 2: "
      f"{sum(len(v) for v in tier2.values()):,}, distinct forms: {len({w for v in tier2.values() for w in v}):,}")
json.dump(tier2, open(OUT / "tier2.json", "w"), ensure_ascii=False, indent=1)
elapsed("tiers 3 and 2")


Unigram EM on the training parts of 110 languages, 24,207,812 words


  EM 1: tokens used 98,490/169,970


  EM 2: tokens used 94,938/169,970


  EM 3: tokens used 94,117/169,970


  EM 4: tokens used 93,638/169,970


| Language | Script | Word unit | Candidate lines/eval pieces | Cut rate (mixed EM, tier 1) | Cut rate (language EM, tier 1) | (a) subwords + words | (b) words + subwords | Order kept | Tier 3 subwords | Tier 2 | Cut rate after both | Target | Word timing |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| English | latin | whitespace | 290,122/754 | 0.18% | 0.18% | 0 + 0 → 0.18% | 0 + 0 → 0.18% | subwords first | 0 | 0 | 0.18% | 0.28% | YODAS2 Whisper onsets |
| French | latin | whitespace | 294,545/766 | 0.59% | 0.55% | 0 + 0 → 0.55% | 0 + 0 → 0.55% | subwords first | 0 | 0 | 0.55% | 1.00% | YODAS2 Whisper onsets |
| Portuguese | latin | whitespace | 284,424/874 | 0.30% | 0.29% | 0 + 0 → 0.29% | 0 + 0 → 0.29% | subwords first | 0 | 0 | 0.29% | 1.00% | YODAS2 Whisper onsets |
| German | latin | whitespace | 309,061/552 | 0.08% | 0.08% | 0 + 0 → 0.08% | 0 + 0 → 0.08% | subwords first | 0 | 0 | 0.08% | 1.00% | YODAS2 Whisper onsets |
| Romanian | latin | whitespace | 212,073/712 | 0.58% | 0.58% | 0 + 0 → 0.58% | 0 + 0 → 0.58% | subwords first | 0 | 0 | 0.58% | 1.00% | YODAS2 Whisper onsets |
| Swedish | latin | whitespace | 258,062/848 | 0.56% | 0.58% | 0 + 0 → 0.58% | 0 + 0 → 0.58% | subwords first | 0 | 0 | 0.58% | 1.00% | YODAS2 Whisper onsets |
| Danish | latin | whitespace | 174,947/765 | 0.87% | 0.89% | 0 + 0 → 0.89% | 0 + 0 → 0.89% | subwords first | 0 | 0 | 0.89% | 1.00% | YODAS2 Whisper onsets |
| Bulgarian | cyrillic | whitespace | 194,173/200 | 5.03% | 5.34% | 3,000 + 0 → 0.96% | 3,750 + 0 → 0.96% | subwords first | 3000 | 0 | 0.96% | 1.00% | YODAS2 Whisper onsets |
| Russian | cyrillic | whitespace | 268,351/747 | 0.51% | 0.49% | 0 + 0 → 0.49% | 0 + 0 → 0.49% | subwords first | 0 | 0 | 0.49% | 1.00% | YODAS2 Whisper onsets |
| Czech | latin | whitespace | 281,843/537 | 2.20% | 2.27% | 2,000 + 0 → 0.61% | 750 + 0 → 0.94% | words first | 0 | 750 | 0.94% | 1.00% | YODAS2 Whisper onsets |
| Greek | greek | whitespace | 262,956/399 | 38.67% | 38.51% | 5,000 + 0 → 0.88% | 20,000 + 1,000 → 0.59% | subwords first | 5000 | 0 | 0.88% | 1.00% | YODAS2 Whisper onsets |
| Ukrainian | cyrillic | whitespace | 260,013/792 | 5.12% | 5.11% | 1,000 + 0 → 0.86% | 9,250 + 0 → 0.99% | subwords first | 1000 | 0 | 0.86% | 1.00% | YODAS2 Whisper onsets |
| Spanish | latin | whitespace | 276,096/580 | 0.24% | 0.23% | 0 + 0 → 0.23% | 0 + 0 → 0.23% | subwords first | 0 | 0 | 0.23% | 1.00% | YODAS2 Whisper onsets |
| Dutch | latin | whitespace | 310,555/849 | 0.30% | 0.30% | 0 + 0 → 0.30% | 0 + 0 → 0.30% | subwords first | 0 | 0 | 0.30% | 1.00% | YODAS2 Whisper onsets |
| Slovak | latin | whitespace | 169,866/435 | 1.66% | 1.65% | 1,000 + 0 → 0.53% | 250 + 0 → 0.86% | words first | 0 | 250 | 0.86% | 1.00% | YODAS2 Whisper onsets |
| Croatian | latin | whitespace | 167,460/499 | 0.49% | 0.50% | 0 + 0 → 0.50% | 0 + 0 → 0.50% | subwords first | 0 | 0 | 0.50% | 1.00% | YODAS2 Whisper onsets |
| Polish | latin | whitespace | 257,228/647 | 0.48% | 0.47% | 0 + 0 → 0.47% | 0 + 0 → 0.47% | subwords first | 0 | 0 | 0.47% | 1.00% | YODAS2 Whisper onsets |
| Lithuanian | latin | whitespace | 162,024/389 | 0.71% | 0.74% | 0 + 0 → 0.74% | 0 + 0 → 0.74% | subwords first | 0 | 0 | 0.74% | 1.00% | YODAS2 Whisper onsets |
| Norwegian Bokmal | latin | whitespace | 241,286/616 | 0.44% | 0.44% | 0 + 0 → 0.44% | 0 + 0 → 0.44% | subwords first | 0 | 0 | 0.44% | 1.00% | YODAS2 Whisper onsets |
| Norwegian Nynorsk | latin | whitespace | 23,992/5,999 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Persian | arabic | whitespace | 185,192/511 | 10.19% | 10.42% | 3,000 + 0 → 0.78% | 7,750 + 0 → 1.00% | subwords first | 3000 | 0 | 0.78% | 1.00% | YODAS2 Whisper onsets |
| Slovenian | latin | whitespace | 162,956/463 | 1.59% | 1.62% | 1,000 + 0 → 0.49% | 250 + 0 → 0.60% | words first | 0 | 250 | 0.60% | 1.00% | YODAS2 Whisper onsets |
| Gujarati | gujarati | whitespace | 4,738/916 | 1.14% | 1.14% | 1,000 + 0 → 0.25% | 250 + 0 → 0.81% | words first | 0 | 250 | 0.81% | 1.00% | approximation, FLEURS recording |
| Latvian | latin | whitespace | 162,961/148 | 2.85% | 2.83% | 1,000 + 0 → 0.66% | 1,500 + 0 → 0.98% | subwords first | 1000 | 0 | 0.66% | 1.00% | YODAS2 Whisper onsets |
| Italian | latin | whitespace | 269,371/890 | 0.31% | 0.31% | 0 + 0 → 0.31% | 0 + 0 → 0.31% | subwords first | 0 | 0 | 0.31% | 1.00% | YODAS2 Whisper onsets |
| Occitan | latin | whitespace | 3,884/961 | 0.62% | 0.62% | 0 + 0 → 0.62% | 0 + 0 → 0.62% | subwords first | 0 | 0 | 0.62% | 1.00% | approximation, FLEURS recording |
| Nepali | devanagari | whitespace | 4,729/170 | 8.83% | 8.83% | 6,000 + 0 → 0.64% | 8,750 + 1,000 → 0.45% | subwords first | 6000 | 0 | 0.64% | 1.00% | YODAS2 Whisper onsets |
| Marathi | devanagari | whitespace | 10,815/946 | 0.90% | 0.90% | 0 + 0 → 0.90% | 0 + 0 → 0.90% | subwords first | 0 | 0 | 0.90% | 1.00% | approximation, FLEURS recording |
| Belarusian | cyrillic | whitespace | 10,914/889 | 0.11% | 0.10% | 0 + 0 → 0.10% | 0 + 0 → 0.10% | subwords first | 0 | 0 | 0.10% | 1.00% | YODAS2 Whisper onsets |
| Serbian | cyrillic | whitespace | 163,147/624 | 2.06% | 2.10% | 1,000 + 0 → 0.97% | 2,750 + 0 → 0.99% | subwords first | 1000 | 0 | 0.97% | 1.00% | YODAS2 Whisper onsets |
| Luxembourgish | latin | whitespace | 3,153/769 | 0.29% | 0.29% | 0 + 0 → 0.29% | 0 + 0 → 0.29% | subwords first | 0 | 0 | 0.29% | 1.00% | approximation, FLEURS recording |
| Venetian | latin | whitespace | 23,994/5,999 | 0.00% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Assamese | bengali | whitespace | 3,807/843 | 0.33% | 0.33% | 0 + 0 → 0.33% | 0 + 0 → 0.33% | subwords first | 0 | 0 | 0.33% | 1.00% | approximation, FLEURS recording |
| Welsh | latin | whitespace | 6,070/325 | 3.24% | 3.25% | 7,000 + 250 → 0.83% | 6,000 + 0 → 0.99% | words first | 0 | 6000 | 0.99% | 1.00% | YODAS2 Whisper onsets |
| Silesian | latin | whitespace | 23,996/6,000 | 0.01% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Asturian | latin | whitespace | 3,084/771 | 0.18% | 0.17% | 0 + 0 → 0.17% | 0 + 0 → 0.17% | subwords first | 0 | 0 | 0.17% | 1.00% | approximation, FLEURS recording |
| Chhattisgarhi | devanagari | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Awadhi | devanagari | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Maithili | devanagari | whitespace | 23,957/5,990 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Bhojpuri | devanagari | whitespace | 24,000/5,998 | 0.02% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, median characters per second |
| Sindhi | arabic | whitespace | 3,931/970 | 2.04% | 2.05% | 7,472 + 59 → 0.09% | 250 + 0 → 0.27% | words first | 0 | 250 | 0.27% | 1.00% | approximation, FLEURS recording |
| Irish | latin | whitespace | 4,099/812 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, FLEURS recording |
| Faroese | latin | whitespace | 24,100/5,987 | 0.01% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Hindi | devanagari | whitespace | 145,642/697 | 26.67% | 26.68% | 6,000 + 0 → 0.74% | 19,500 + 0 → 0.62% | subwords first | 6000 | 0 | 0.74% | 1.00% | YODAS2 Whisper onsets |
| Punjabi | gurmukhi | whitespace | 2,455/169 | 2.90% | 2.90% | 6,704 + 0 → 1.17% | 750 + 5,992 → 1.13% | words first | 5992 | 750 | 1.13% | 1.00% | YODAS2 Whisper onsets |
| Bengali | bengali | whitespace | 169,805/20,000 | 0.08% | 0.08% | 0 + 0 → 0.08% | 0 + 0 → 0.08% | subwords first | 0 | 0 | 0.08% | 1.00% | approximation, FLEURS recording |
| Oriya | oriya | whitespace | 1,921/472 | 1.35% | 1.34% | 1,000 + 0 → 0.09% | 250 + 0 → 0.88% | words first | 0 | 250 | 0.88% | 1.00% | approximation, FLEURS recording |
| Tajik | cyrillic | whitespace | 2,673/628 | 0.15% | 0.16% | 0 + 0 → 0.16% | 0 + 0 → 0.16% | subwords first | 0 | 0 | 0.16% | 1.00% | approximation, FLEURS recording |
| Eastern Yiddish | hebrew | whitespace | 23,606/5,892 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Lombard | latin | whitespace | 23,945/5,987 | 0.01% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Ligurian | latin | whitespace | 23,965/5,992 | 0.02% | 0.03% | 0 + 0 → 0.03% | 0 + 0 → 0.03% | subwords first | 0 | 0 | 0.03% | 1.00% | approximation, median characters per second |
| Sicilian | latin | whitespace | 23,998/6,000 | 0.00% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Friulian | latin | whitespace | 19,283/4,821 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Sardinian | latin | whitespace | 23,976/5,994 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Galician | latin | whitespace | 167,758/355 | 0.46% | 0.50% | 0 + 0 → 0.50% | 0 + 0 → 0.50% | subwords first | 0 | 0 | 0.50% | 1.00% | YODAS2 Whisper onsets |
| Catalan | latin | whitespace | 208,701/795 | 0.82% | 0.83% | 0 + 0 → 0.83% | 0 + 0 → 0.83% | subwords first | 0 | 0 | 0.83% | 1.00% | YODAS2 Whisper onsets |
| Icelandic | latin | whitespace | 163,723/484 | 7.43% | 7.52% | 3,000 + 0 → 0.85% | 7,750 + 0 → 0.99% | subwords first | 3000 | 0 | 0.85% | 1.00% | YODAS2 Whisper onsets |
| Tosk Albanian | latin | whitespace | 160,881/20,000 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Limburgish | latin | whitespace | 23,999/6,000 | 0.00% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Dari | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Afrikaans | latin | whitespace | 50,978/12,561 | 0.17% | 0.16% | 0 + 0 → 0.16% | 0 + 0 → 0.16% | subwords first | 0 | 0 | 0.16% | 1.00% | approximation, FLEURS recording |
| Macedonian | cyrillic | whitespace | 165,067/216 | 2.67% | 2.98% | 1,000 + 0 → 0.53% | 500 + 0 → 0.89% | words first | 0 | 500 | 0.89% | 1.00% | YODAS2 Whisper onsets |
| Sinhala | sinhala | whitespace | 162,404/149 | 14.16% | 14.16% | 15,000 + 0 → 0.95% | 8,750 + 7,000 → 0.85% | subwords first | 15000 | 0 | 0.95% | 1.00% | YODAS2 Whisper onsets |
| Urdu | arabic | whitespace | 51,088/189 | 20.83% | 21.52% | 3,000 + 0 → 0.85% | 9,000 + 0 → 0.76% | subwords first | 3000 | 0 | 0.85% | 1.00% | YODAS2 Whisper onsets |
| Magahi | devanagari | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Bosnian | latin | whitespace | 166,916/280 | 0.59% | 0.58% | 0 + 0 → 0.58% | 0 + 0 → 0.58% | subwords first | 0 | 0 | 0.58% | 1.00% | YODAS2 Whisper onsets |
| Armenian | armenian | whitespace | 7,626/1,695 | 6.00% | 5.98% | 5,000 + 250 → 0.08% | 10,750 + 0 → 0.62% | subwords first | 5000 | 250 | 0.08% | 1.00% | approximation, FLEURS recording |
| Simplified Chinese | han | Unigram units | 163,649/20,000 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 0.20% | approximation, FLEURS recording |
| Traditional Chinese | han | Unigram units | 137,091/20,000 | 0.10% | 0.10% | 0 + 0 → 0.10% | 0 + 0 → 0.10% | subwords first | 0 | 0 | 0.10% | 0.20% | approximation, median characters per second |
| Cantonese | han | Unigram units | 2,496/624 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, FLEURS recording |
| Burmese | myanmar | Unigram units | 3,457/865 | 9.68% | 9.68% | 6,449 + 0 → 0.98% | 4,500 + 0 → 0.92% | words first | 0 | 4500 | 0.92% | 1.00% | approximation, FLEURS recording |
| Standard Arabic | arabic | whitespace | 196,292/244 | 0.59% | 0.65% | 0 + 0 → 0.65% | 0 + 0 → 0.65% | subwords first | 0 | 0 | 0.65% | 1.00% | YODAS2 Whisper onsets |
| Najdi Arabic | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Levantine Arabic | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Egyptian Arabic | arabic | whitespace | 23,877/5,970 | 0.01% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, median characters per second |
| Moroccan Arabic | arabic | whitespace | 23,964/5,992 | 0.01% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, median characters per second |
| Mesopotamian Arabic | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Taizzi-Adeni Arabic | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Tunisian Arabic | arabic | - | - | - | - | - | - | - | 0 | 0 | - | - | no data |
| Hebrew | hebrew | whitespace | 320,019/660 | 0.37% | 0.36% | 0 + 0 → 0.36% | 0 + 0 → 0.36% | subwords first | 0 | 0 | 0.36% | 1.00% | YODAS2 Whisper onsets |
| Maltese | latin | whitespace | 3,380/845 | 0.01% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, FLEURS recording |
| Indonesian | latin | whitespace | 200,944/753 | 0.13% | 0.16% | 0 + 0 → 0.16% | 0 + 0 → 0.16% | subwords first | 0 | 0 | 0.16% | 1.00% | YODAS2 Whisper onsets |
| Malay | latin | whitespace | 174,681/281 | 0.31% | 0.33% | 0 + 0 → 0.33% | 0 + 0 → 0.33% | subwords first | 0 | 0 | 0.33% | 1.00% | YODAS2 Whisper onsets |
| Tagalog | latin | whitespace | 18,064/4,516 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, FLEURS recording |
| Cebuano | latin | whitespace | 3,221/806 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, FLEURS recording |
| Javanese | latin | whitespace | 3,991/815 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, FLEURS recording |
| Sundanese | latin | whitespace | 24,018/5,997 | 0.02% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, median characters per second |
| Minangkabau | latin | whitespace | 23,982/5,996 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Balinese | latin | whitespace | 23,999/6,000 | 0.08% | 0.08% | 0 + 0 → 0.08% | 0 + 0 → 0.08% | subwords first | 0 | 0 | 0.08% | 1.00% | approximation, median characters per second |
| Banjar | latin | whitespace | 23,972/5,993 | 0.01% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Pangasinan | latin | whitespace | 10,826/2,707 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Iloko | latin | whitespace | 24,000/6,000 | 0.01% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, median characters per second |
| Waray | latin | whitespace | 23,994/5,999 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Tamil | tamil | whitespace | 50,196/82 | 51.43% | 51.04% | 4,000 + 0 → 0.76% | 16,000 + 1,000 → 0.03% | subwords first | 4000 | 0 | 0.76% | 1.00% | YODAS2 Whisper onsets |
| Telugu | telugu | whitespace | 30,851/6,700 | 0.33% | 0.30% | 0 + 0 → 0.30% | 0 + 0 → 0.30% | subwords first | 0 | 0 | 0.30% | 1.00% | approximation, FLEURS recording |
| Kannada | kannada | whitespace | 4,293/698 | 0.31% | 0.26% | 0 + 0 → 0.26% | 0 + 0 → 0.26% | subwords first | 0 | 0 | 0.26% | 1.00% | approximation, FLEURS recording |
| Malayalam | malayalam | whitespace | 172,549/20,000 | 1.54% | 1.47% | 1,000 + 0 → 0.23% | 4,750 + 0 → 1.00% | subwords first | 1000 | 0 | 0.23% | 1.00% | approximation, FLEURS recording |
| Turkish | latin | whitespace | 232,030/816 | 0.19% | 0.20% | 0 + 0 → 0.20% | 0 + 0 → 0.20% | subwords first | 0 | 0 | 0.20% | 1.00% | YODAS2 Whisper onsets |
| North Azerbaijani | latin | whitespace | 3,190/269 | 3.71% | 4.09% | 1,000 + 0 → 0.88% | 6,000 + 1,000 → 0.41% | subwords first | 1000 | 0 | 0.88% | 1.00% | YODAS2 Whisper onsets |
| Northern Uzbek | latin | whitespace | 5,042/58 | 0.55% | 0.55% | 0 + 0 → 0.55% | 0 + 0 → 0.55% | subwords first | 0 | 0 | 0.55% | 1.00% | YODAS2 Whisper onsets |
| Kazakh | cyrillic | whitespace | 8,038/1,705 | 0.01% | 0.02% | 0 + 0 → 0.02% | 0 + 0 → 0.02% | subwords first | 0 | 0 | 0.02% | 1.00% | approximation, FLEURS recording |
| Bashkir | cyrillic | whitespace | 24,003/5,998 | 0.05% | 0.18% | 0 + 0 → 0.18% | 0 + 0 → 0.18% | subwords first | 0 | 0 | 0.18% | 1.00% | approximation, median characters per second |
| Tatar | cyrillic | whitespace | 23,948/5,979 | 0.02% | 0.14% | 0 + 0 → 0.14% | 0 + 0 → 0.14% | subwords first | 0 | 0 | 0.14% | 1.00% | approximation, median characters per second |
| Thai | thai | Unigram units | 163,192/20,000 | 0.46% | 0.56% | 0 + 0 → 0.56% | 0 + 0 → 0.56% | subwords first | 0 | 0 | 0.56% | 1.00% | approximation, FLEURS recording |
| Lao | lao | Unigram units | 1,924/481 | 4.71% | 4.59% | 2,000 + 0 → 0.73% | 3,000 + 0 → 0.93% | subwords first | 2000 | 0 | 0.73% | 1.00% | approximation, FLEURS recording |
| Finnish | latin | whitespace | 298,087/821 | 1.33% | 1.35% | 1,000 + 0 → 0.48% | 250 + 0 → 0.37% | words first | 0 | 250 | 0.37% | 1.00% | YODAS2 Whisper onsets |
| Estonian | latin | whitespace | 172,613/777 | 0.98% | 0.97% | 0 + 0 → 0.97% | 0 + 0 → 0.97% | subwords first | 0 | 0 | 0.97% | 1.00% | YODAS2 Whisper onsets |
| Hungarian | latin | whitespace | 277,358/897 | 0.36% | 0.38% | 0 + 0 → 0.38% | 0 + 0 → 0.38% | subwords first | 0 | 0 | 0.38% | 1.00% | YODAS2 Whisper onsets |
| Vietnamese | latin | whitespace | 186,305/454 | 0.81% | 0.81% | 0 + 0 → 0.81% | 0 + 0 → 0.81% | subwords first | 0 | 0 | 0.81% | 1.00% | YODAS2 Whisper onsets |
| Khmer | khmer | Unigram units | 2,217/555 | 3.04% | 2.98% | 5,861 + 26 → 0.07% | 2,250 + 0 → 0.96% | words first | 0 | 2250 | 0.96% | 1.00% | approximation, FLEURS recording |
| Japanese | kana | Unigram units | 162,566/20,000 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 0.20% | approximation, FLEURS recording |
| Korean | hangul | whitespace | 2,906,900/578 | 0.43% | 0.45% | 8,000 + 0 → 0.23% | 3,750 + 0 → 0.23% | words first | 0 | 3750 | 0.23% | 0.23% | YODAS2 Whisper onsets |
| Georgian | georgian | whitespace | 155,917/858 | 0.47% | 0.47% | 0 + 0 → 0.47% | 0 + 0 → 0.47% | subwords first | 0 | 0 | 0.47% | 1.00% | YODAS2 Whisper onsets |
| Basque | latin | whitespace | 159,948/461 | 0.43% | 0.49% | 0 + 0 → 0.49% | 0 + 0 → 0.49% | subwords first | 0 | 0 | 0.49% | 1.00% | YODAS2 Whisper onsets |
| Haitian | latin | whitespace | 24,674/5,988 | 0.14% | 0.15% | 0 + 0 → 0.15% | 0 + 0 → 0.15% | subwords first | 0 | 0 | 0.15% | 1.00% | approximation, median characters per second |
| Papiamento | latin | whitespace | 16,328/4,083 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, median characters per second |
| Kabuverdianu | latin | whitespace | 3,156/789 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, FLEURS recording |
| Tok Pisin | latin | whitespace | 2,568/643 | 0.04% | 0.01% | 0 + 0 → 0.01% | 0 + 0 → 0.01% | subwords first | 0 | 0 | 0.01% | 1.00% | approximation, median characters per second |
| Swahili | latin | whitespace | 3,014/754 | 0.00% | 0.00% | 0 + 0 → 0.00% | 0 + 0 → 0.00% | subwords first | 0 | 0 | 0.00% | 1.00% | approximation, FLEURS recording |

Tier 3 subwords summed over languages: 65,992; tier 2: 20,250, distinct forms: 20,170
[tiers 3 and 2] 1,645 s since the start


### Why words are cut

For every language measured on real onsets, each word still cut with the tier 2 forms alone (no subwords, counted with the
mixed EM) is described by three things: the gap from its
onset to the next word's onset, its token count, and whether its timing is physically plausible. A word or a gap shorter than
30 ms per character cannot be spoken and marks an alignment error. A cut needs the next onset to come before the word's tokens
end, so the gap and the token count together tell whether a cut comes from fast speech, from many tokens, or from the aligner.

In [13]:
def cut_words(rows, added):
    # Every cut word with its token count, the gap to the next onset and its own span, in seconds, placed as cut_rate does.
    found = []
    for row in rows:
        words = [(spoken_part(unit), start, end) for unit, start, end in zip(row["text"].split(), row["starts"], row["ends"])]
        words = [item for item in words if item[0]]
        frames = 1 + math.ceil(row["seconds"] * FRAMES_PER_SECOND)
        previous = None
        for k, (word, start, end) in enumerate(words):
            onset, tokens = 1 + int(start * FRAMES_PER_SECOND), token_count(word, added)
            gap = words[k + 1][1] - start if k + 1 < len(words) else None
            if onset + tokens > frames:
                found.append((word, tokens, gap, end - start))
                continue
            if previous is not None and onset < previous[0]:
                found.append(previous[1])
            previous = (onset + tokens, (word, tokens, gap, end - start))
    return found


def share(items, test):
    return f"{sum(1 for item in items if test(item)) / max(len(items), 1):.0%}"


cause_rows, examples = [], {}
for language in QWEN3_LANGUAGES:
    if scripts[language.name] in NO_SPACE_SCRIPTS or language.name not in splits:
        continue
    subset = f"{YODAS_CODE.get(language.name, language.wikipedia)}000"
    rows = [row for row in yodas_timings.get(subset, []) if row["loss"] <= LOSS_LIMIT and squeezed_share(row) <= MAX_SQUEEZED]
    if sum(len(row["text"].split()) for row in rows) < MIN_REAL_WORDS:
        continue
    found = cut_words(rows, set(tier2[language.name]))
    timed = [item for item in found if item[2] is not None]
    implausible = lambda item: item[3] / len(item[0]) < 0.03 or (item[2] is not None and item[2] / len(item[0]) < 0.03)
    cause_rows.append([language.name, len(found),
                       share(timed, lambda i: i[2] < 0.02), share(timed, lambda i: 0.02 <= i[2] < 0.08),
                       share(timed, lambda i: 0.08 <= i[2] < 0.16), share(timed, lambda i: i[2] >= 0.16),
                       share(found, lambda i: i[1] == 1), share(found, lambda i: i[1] == 2), share(found, lambda i: i[1] >= 3),
                       share(found, implausible)])
    examples[language.name] = [f"`{word}` {tokens} tok, next {gap * 1000:.0f} ms" for word, tokens, gap, _ in timed[:6]]
table(cause_rows, ["Language", "Cut words", "Next onset < 20 ms", "20-80 ms", "80-160 ms", ">= 160 ms",
                   "1 token", "2 tokens", "3+ tokens", "Implausible timing"])
for name in ("English", "French", "Korean", "Hindi", "Ukrainian", "Tamil", "Greek"):
    if name in examples:
        print(name + ":", "; ".join(examples[name]))

| Language | Cut words | Next onset < 20 ms | 20-80 ms | 80-160 ms | >= 160 ms | 1 token | 2 tokens | 3+ tokens | Implausible timing |
|---|---|---|---|---|---|---|---|---|---|
| English | 532 | 42% | 37% | 21% | 1% | 53% | 41% | 5% | 88% |
| French | 1525 | 36% | 28% | 26% | 10% | 52% | 31% | 17% | 87% |
| Portuguese | 708 | 40% | 33% | 22% | 5% | 58% | 34% | 9% | 84% |
| German | 307 | 63% | 14% | 15% | 8% | 59% | 17% | 24% | 95% |
| Romanian | 348 | 29% | 21% | 30% | 21% | 33% | 31% | 36% | 63% |
| Swedish | 687 | 30% | 32% | 26% | 12% | 51% | 29% | 21% | 78% |
| Danish | 1315 | 32% | 32% | 25% | 11% | 53% | 30% | 18% | 79% |
| Bulgarian | 599 | 27% | 16% | 27% | 30% | 17% | 31% | 52% | 62% |
| Russian | 304 | 30% | 15% | 20% | 34% | 32% | 20% | 48% | 57% |
| Czech | 480 | 47% | 22% | 8% | 23% | 50% | 9% | 41% | 68% |
| Greek | 7247 | 6% | 8% | 27% | 59% | 1% | 16% | 83% | 24% |
| Ukrainian | 2098 | 16% | 8% | 24% | 52% | 9% | 21% | 70% | 30% |
| Spanish | 311 | 41% | 43% | 11% | 5% | 73% | 18% | 9% | 83% |
| Dutch | 806 | 41% | 30% | 22% | 7% | 54% | 32% | 13% | 87% |
| Slovak | 296 | 36% | 25% | 13% | 27% | 44% | 13% | 43% | 59% |
| Croatian | 247 | 33% | 25% | 21% | 20% | 38% | 30% | 32% | 62% |
| Polish | 506 | 43% | 18% | 22% | 18% | 38% | 25% | 37% | 78% |
| Lithuanian | 225 | 40% | 10% | 24% | 26% | 22% | 36% | 42% | 63% |
| Norwegian Bokmal | 482 | 26% | 44% | 20% | 11% | 63% | 23% | 15% | 73% |
| Persian | 3130 | 10% | 12% | 36% | 42% | 5% | 41% | 54% | 24% |
| Slovenian | 313 | 41% | 36% | 12% | 11% | 64% | 17% | 20% | 78% |
| Latvian | 270 | 22% | 14% | 26% | 38% | 12% | 30% | 58% | 47% |
| Italian | 416 | 30% | 40% | 20% | 10% | 60% | 20% | 20% | 74% |
| Nepali | 274 | 7% | 5% | 8% | 79% | 4% | 5% | 91% | 15% |
| Belarusian | 396 | 75% | 19% | 4% | 2% | 77% | 15% | 8% | 90% |
| Serbian | 719 | 13% | 14% | 36% | 37% | 19% | 32% | 49% | 31% |
| Welsh | 659 | 41% | 35% | 16% | 7% | 59% | 25% | 16% | 78% |
| Hindi | 8538 | 1% | 5% | 31% | 63% | 0% | 23% | 77% | 6% |
| Punjabi | 148 | 8% | 4% | 22% | 66% | 2% | 7% | 91% | 12% |
| Galician | 192 | 34% | 32% | 27% | 7% | 48% | 35% | 17% | 69% |
| Catalan | 1017 | 32% | 29% | 25% | 15% | 47% | 28% | 25% | 74% |
| Icelandic | 2139 | 11% | 16% | 46% | 27% | 7% | 49% | 44% | 38% |
| Macedonian | 285 | 54% | 16% | 9% | 21% | 58% | 8% | 34% | 80% |
| Sinhala | 468 | 17% | 10% | 17% | 57% | 1% | 7% | 92% | 40% |
| Urdu | 2435 | 8% | 9% | 47% | 36% | 1% | 42% | 57% | 17% |
| Bosnian | 144 | 19% | 25% | 40% | 16% | 39% | 38% | 24% | 48% |
| Standard Arabic | 196 | 49% | 22% | 18% | 11% | 39% | 34% | 27% | 83% |
| Hebrew | 632 | 61% | 20% | 10% | 9% | 49% | 28% | 23% | 84% |
| Indonesian | 391 | 46% | 13% | 23% | 18% | 22% | 38% | 40% | 93% |
| Malay | 118 | 37% | 17% | 36% | 10% | 21% | 57% | 22% | 84% |
| Tamil | 1645 | 0% | 0% | 3% | 97% | 0% | 0% | 100% | 4% |
| Turkish | 117 | 37% | 14% | 32% | 16% | 26% | 41% | 32% | 65% |
| North Azerbaijani | 434 | 16% | 5% | 14% | 65% | 5% | 16% | 79% | 25% |
| Northern Uzbek | 17 | 18% | 6% | 35% | 41% | 6% | 29% | 65% | 24% |
| Finnish | 573 | 46% | 19% | 16% | 19% | 32% | 20% | 49% | 86% |
| Estonian | 532 | 19% | 15% | 45% | 22% | 12% | 52% | 36% | 50% |
| Hungarian | 114 | 23% | 23% | 36% | 18% | 23% | 50% | 27% | 51% |
| Vietnamese | 473 | 14% | 33% | 45% | 8% | 40% | 45% | 15% | 55% |
| Korean | 102 | 52% | 11% | 17% | 20% | 34% | 22% | 44% | 63% |
| Georgian | 446 | 5% | 27% | 45% | 23% | 20% | 52% | 28% | 65% |
| Basque | 240 | 11% | 19% | 61% | 9% | 7% | 73% | 20% | 69% |

English: `this` 1 tok, next 0 ms; `It's` 2 tok, next 120 ms; `people` 1 tok, next 20 ms; `of` 1 tok, next 0 ms; `2009` 4 tok, next 260 ms; `This` 1 tok, next 0 ms
French: `C'est` 2 tok, next 120 ms; `je` 1 tok, next 40 ms; `voit` 2 tok, next 80 ms; `voit` 2 tok, next 120 ms; `as` 1 tok, next 40 ms; `qu'est` 2 tok, next 100 ms
Korean: `왜` 1 tok, next 0 ms; `보여` 2 tok, next 100 ms; `분해를` 3 tok, next 180 ms; `다행` 2 tok, next 100 ms; `전원을` 3 tok, next 200 ms; `그러인더` 3 tok, next 220 ms
Hindi: `कपड़े` 4 tok, next 160 ms; `के` 2 tok, next 40 ms; `अलावा` 5 tok, next 320 ms; `को` 2 tok, next 80 ms; `व्यक्त` 6 tok, next 200 ms; `हैं` 3 tok, next 160 ms
Ukrainian: `Йосип` 3 tok, next 0 ms; `водій` 3 tok, next 0 ms; `вантажівки` 6 tok, next 0 ms; `які` 3 tok, next 120 ms; `Віталій` 5 tok, next 0 ms; `водій` 3 tok, next 0 ms
Tamil: `என்னால்` 7 tok, next 140 ms; `ஆன` 2 tok, next 0 ms; `அந்த` 4 tok, next 80 ms; `உறுதிமொழியைப்` 13 tok, next 80 ms; `பின்பற்ற` 8 tok, next 300 ms; `முயற்சிக்கிறேன்` 15 to

### Check: the approximation against real word timings (KsponSpeech and FLEURS)

The cut approximation places words by their share of characters. The Whisper word timings above give real onsets. On
KsponSpeech (Korean spontaneous speech) the cut rate is measured twice, with the real onsets and with the approximation, and
Korean tier 2 is selected again on the real onsets with the same candidates and stopping rule as above. None of the Korean tier 2
data above comes from KsponSpeech. Then for every FLEURS language the cut rate with the current vocabulary is measured both ways.
For scripts written without spaces a whitespace unit is a whole phrase, so their Whisper onsets do not time words and they are
listed without a comparison.

In [14]:
import numpy as np

# Each record: {"id", "text", "seconds", "starts", "ends", "loss"}, one onset and one end per whitespace word of "text".
aligned = word_timings["ksponspeech"]


real = [real_layout(row["text"], row["starts"], row["seconds"], row["ends"]) for row in aligned]
approximate = [layout(row["text"].split(), row["seconds"]) for row in aligned]

counts = unit_counts["Korean"]
candidates = [(count * (token_count(word) - 1), word) for word, count in counts.items() if count >= 2 and token_count(word) >= 2]
candidates = [word for _, word in sorted(candidates, reverse=True)]
current = cut_rate(real)
added, steps = set(), [(0, current)]
while current >= 0.01 and len(added) < 20_000 and len(added) < len(candidates):
    batch = candidates[len(added):len(added) + 250]
    trial = cut_rate(real, added | set(batch))
    if current - trial <= 0.0001:
        break
    added |= set(batch)
    current = trial
    steps.append((len(added), current))
korean_real_tier2 = [word for word in candidates if word in added]

table([
    ["Utterances", f"{len(aligned):,}"],
    ["Words", f"{sum(len(words) for words, *_ in real):,}"],
    ["Cut rate, real onsets", f"{cut_rate(real):.2%}"],
    ["Cut rate, approximation", f"{cut_rate(approximate):.2%}"],
    ["Korean tier 2 selected on real onsets", f"{len(korean_real_tier2):,}"],
    ["Cut rate, real onsets, after that tier 2", f"{current:.2%}"],
], ["KsponSpeech (Whisper onsets)", "Value"])
print("Cut rate on real onsets as tier 2 grows:", ", ".join(f"{size:,}: {rate:.2%}" for size, rate in steps[::4] + steps[-1:]))


everything = {word for words, *_ in real for word in words}
floor = cut_rate(real, everything)
rows = []
for label, added in (("current vocabulary", ()), (f"with the {len(korean_real_tier2):,} tier 2 forms", set(korean_real_tier2))):
    placed = placements(real, added)
    cut_words = [(word, tokens) for word, tokens, cut, *_ in placed if cut]
    unannounced = sum(1 for _, _, _, bare, _ in placed if bare)
    one = sum(1 for _, tokens in cut_words if tokens == 1)
    seen = sum(1 for word, tokens in cut_words if tokens > 1 and counts.get(word, 0) >= 2)
    unseen = sum(1 for word, tokens in cut_words if tokens > 1 and counts.get(word, 0) < 2)
    total = max(sum(1 for *_, ok in placed if ok), 1)
    rows.append([label, f"{unannounced / total:.2%}", f"{len(cut_words) / total:.2%}", f"{one / total:.2%}", f"{seen / total:.2%}", f"{unseen / total:.2%}"])
table(rows, ["Real onsets", "Written without EPAD", "Cut", "Cut, already one token", "Cut, several tokens, seen twice in training",
             "Cut, several tokens, not seen twice in training"])
print(f"Cut rate on real onsets if every word were one token: {floor:.2%}")


fleurs_language = {language.fleurs: language for language in QWEN3_LANGUAGES if language.fleurs}
rows = []
for code, records in word_timings.items():
    if code == "ksponspeech":
        continue
    language = fleurs_language[code]
    if scripts[language.name] in NO_SPACE_SCRIPTS:
        rows.append([language.name, len(records), "-", "-", "no spaces"])
        continue
    real_layouts = [real_layout(row["text"], row["starts"], row["seconds"], row["ends"]) for row in records]
    approximate_layouts = [layout(row["text"].split(), row["seconds"]) for row in records]
    rows.append([language.name, len(records), f"{cut_rate(real_layouts):.2%}", f"{cut_rate(approximate_layouts):.2%}", ""])
table(rows, ["FLEURS language", "Utterances", "Cut rate, Whisper onsets", "Cut rate, approximation", "Note"])


# Korean on three kinds of speech, with the alignment loss used to leave out pieces whose subtitles do not match.
loss_limit = float(np.percentile([row["loss"] for row in word_timings["ksponspeech"]], 95))
compare = [("KsponSpeech (conversation)", word_timings["ksponspeech"]), ("FLEURS (read)", word_timings["ko_kr"]),
           ("YODAS2 ko000, all pieces", yodas_timings["ko000"]),
           (f"YODAS2 ko000, loss up to {loss_limit:.2f}", [row for row in yodas_timings["ko000"] if row["loss"] <= loss_limit])]
rows = []
for label, records in compare:
    layouts = [real_layout(row["text"], row["starts"], row["seconds"], row["ends"]) for row in records]
    losses = sorted(row["loss"] for row in records)
    gaps = [s2 - s1 for row in records for s1, s2 in zip(row["starts"], row["starts"][1:])]
    rows.append([label, len(records), sum(len(words) for words, *_ in layouts), f"{losses[len(losses) // 2]:.2f}",
                 f"{np.mean(np.array(gaps) < 0.16):.2%}", f"{cut_rate(layouts):.2%}"])
table(rows, ["Korean speech", "Pieces", "Words", "Median alignment loss", "Onset gaps under 160 ms", "Cut rate, Whisper onsets"])

| KsponSpeech (Whisper onsets) | Value |
|---|---|
| Utterances | 6,000 |
| Words | 47,809 |
| Cut rate, real onsets | 0.76% |
| Cut rate, approximation | 0.00% |
| Korean tier 2 selected on real onsets | 0 |
| Cut rate, real onsets, after that tier 2 | 0.76% |

Cut rate on real onsets as tier 2 grows: 0: 0.76%, 0: 0.76%


| Real onsets | Written without EPAD | Cut | Cut, already one token | Cut, several tokens, seen twice in training | Cut, several tokens, not seen twice in training |
|---|---|---|---|---|---|
| current vocabulary | 8.73% | 0.76% | 0.06% | 0.67% | 0.03% |
| with the 0 tier 2 forms | 8.73% | 0.76% | 0.06% | 0.67% | 0.03% |

Cut rate on real onsets if every word were one token: 0.07%


| FLEURS language | Utterances | Cut rate, Whisper onsets | Cut rate, approximation | Note |
|---|---|---|---|---|
| French | 200 | 0.72% | 0.36% |  |
| Portuguese | 200 | 0.14% | 0.05% |  |
| German | 198 | 0.03% | 0.03% |  |
| Romanian | 200 | 0.65% | 0.27% |  |
| Swedish | 200 | 0.08% | 0.05% |  |
| Danish | 199 | 0.29% | 0.08% |  |
| Bulgarian | 199 | 3.06% | 1.08% |  |
| Russian | 200 | 0.34% | 0.14% |  |
| Czech | 200 | 0.52% | 0.14% |  |
| English | 200 | 0.05% | 0.26% |  |
| Greek | 200 | 54.84% | 20.21% |  |
| Ukrainian | 199 | 4.35% | 0.42% |  |
| Spanish | 200 | 0.15% | 0.00% |  |
| Dutch | 171 | 0.32% | 0.05% |  |
| Slovak | 200 | 0.88% | 0.08% |  |
| Croatian | 200 | 0.58% | 0.03% |  |
| Polish | 200 | 0.68% | 0.75% |  |
| Lithuanian | 200 | 1.21% | 0.43% |  |
| Norwegian Bokmal | 161 | 0.30% | 0.03% |  |
| Persian | 196 | 2.26% | 0.00% |  |
| Slovenian | 200 | 0.50% | 0.19% |  |
| Gujarati | 188 | 28.91% | 1.58% |  |
| Latvian | 200 | 0.70% | 0.00% |  |
| Italian | 197 | 0.07% | 0.00% |  |
| Occitan | 199 | 0.12% | 0.00% |  |
| Nepali | 199 | 21.71% | 1.02% |  |
| Marathi | 199 | 19.52% | 0.50% |  |
| Belarusian | 199 | 1.82% | 0.10% |  |
| Serbian | 200 | 3.76% | 0.00% |  |
| Luxembourgish | 200 | 1.58% | 0.55% |  |
| Assamese | 192 | 18.86% | 3.84% |  |
| Welsh | 198 | 0.70% | 0.02% |  |
| Asturian | 200 | 0.60% | 0.29% |  |
| Sindhi | 200 | 6.52% | 2.03% |  |
| Irish | 200 | 0.65% | 0.00% |  |
| Hindi | 200 | 19.75% | 0.53% |  |
| Punjabi | 199 | 24.17% | 3.60% |  |
| Bengali | 191 | 14.36% | 0.76% |  |
| Oriya | 167 | 23.68% | 1.57% |  |
| Tajik | 195 | 1.25% | 0.00% |  |
| Galician | 200 | 0.60% | 0.99% |  |
| Catalan | 199 | 0.70% | 0.15% |  |
| Icelandic | 36 | 1.62% | 0.00% |  |
| Afrikaans | 198 | 0.42% | 0.48% |  |
| Macedonian | 200 | 1.34% | 0.31% |  |
| Urdu | 200 | 15.33% | 0.26% |  |
| Bosnian | 199 | 0.29% | 0.00% |  |
| Armenian | 199 | 38.30% | 9.99% |  |
| Simplified Chinese | 200 | - | - | no spaces |
| Cantonese | 199 | - | - | no spaces |
| Burmese | 116 | - | - | no spaces |
| Standard Arabic | 199 | 0.06% | 0.00% |  |
| Hebrew | 200 | 0.28% | 0.68% |  |
| Maltese | 199 | 1.36% | 0.00% |  |
| Indonesian | 200 | 0.03% | 0.00% |  |
| Malay | 200 | 0.00% | 0.05% |  |
| Tagalog | 192 | 0.07% | 0.00% |  |
| Cebuano | 196 | 0.15% | 0.00% |  |
| Javanese | 199 | 0.18% | 0.00% |  |
| Tamil | 197 | 43.33% | 17.56% |  |
| Telugu | 194 | 35.40% | 3.92% |  |
| Kannada | 199 | 17.30% | 0.56% |  |
| Malayalam | 181 | 21.69% | 3.38% |  |
| Turkish | 200 | 0.10% | 0.00% |  |
| North Azerbaijani | 198 | 4.61% | 0.28% |  |
| Northern Uzbek | 200 | 0.93% | 0.25% |  |
| Kazakh | 197 | 5.16% | 0.00% |  |
| Thai | 198 | - | - | no spaces |
| Lao | 161 | - | - | no spaces |
| Finnish | 200 | 0.48% | 0.03% |  |
| Estonian | 200 | 0.13% | 0.00% |  |
| Hungarian | 198 | 0.71% | 0.28% |  |
| Vietnamese | 199 | 0.45% | 0.00% |  |
| Khmer | 188 | - | - | no spaces |
| Japanese | 200 | - | - | no spaces |
| Korean | 200 | 0.07% | 0.00% |  |
| Georgian | 175 | 49.39% | 8.18% |  |
| Kabuverdianu | 196 | 0.14% | 0.00% |  |
| Swahili | 199 | 0.26% | 0.00% |  |

| Korean speech | Pieces | Words | Median alignment loss | Onset gaps under 160 ms | Cut rate, Whisper onsets |
|---|---|---|---|---|---|
| KsponSpeech (conversation) | 6000 | 47809 | 0.43 | 10.40% | 0.76% |
| FLEURS (read) | 200 | 3039 | 0.10 | 0.63% | 0.07% |
| YODAS2 ko000, all pieces | 1000 | 21633 | 0.93 | 18.72% | 0.56% |
| YODAS2 ko000, loss up to 1.61 | 588 | 17761 | 0.39 | 4.30% | 0.43% |

## 6. Tier 3: subwords and non-standard characters

Tier 3 holds the subwords section 5 chose for each language, and the non-standard characters (no single token in the
vocabulary) seen three or more times in the spoken training part, but only when they belong to one of the language's own
scripts. Characters of other scripts, format characters (Cf), the replacement characters `�` and `￼`, control characters,
private-use characters and whitespace are never added.

In [15]:
tier3, tier3_rows = {}, []
for language in QWEN3_LANGUAGES:
    counts = unit_counts.get(language.name)
    if not counts:
        tier3[language.name] = []
        continue
    subwords = subwords_of[language.name]
    known = vocabulary | {"▁" + word for word in tier2[language.name]} | set(subwords)
    own_scripts = language_scripts(language)
    characters = collections.Counter()
    for word, count in counts.items():
        for ch in word:
            characters[ch] += count
    outside = sorted(ch for ch, count in characters.items()
                     if count >= 3 and ch not in known and not broken(ch) and script_of(ch) in own_scripts)
    tier3[language.name] = subwords + outside
    if subwords or outside:
        tier3_rows.append([language.name, len(subwords), len(outside), "".join(outside[:20])])

table(tier3_rows, ["Language", "Tier 3 subwords", "Tier 3 non-standard characters", "Examples of non-standard characters"])
print(f"Tier 3 summed over languages: {sum(len(v) for v in tier3.values()):,}")
json.dump(tier3, open(OUT / "tier3.json", "w"), ensure_ascii=False, indent=1)
elapsed("tier 3")

| Language | Tier 3 subwords | Tier 3 non-standard characters | Examples of non-standard characters |
|---|---|---|---|
| Bulgarian | 3000 | 0 |  |
| Greek | 5000 | 16 | ἆἍἜἧἳὌὢὤᾐᾖᾠᾤῂῒῠῤ |
| Ukrainian | 1000 | 0 |  |
| Spanish | 0 | 3 | Ʉɇꞌ |
| Norwegian Nynorsk | 0 | 1 | ḗ |
| Persian | 3000 | 0 |  |
| Latvian | 1000 | 0 |  |
| Italian | 0 | 1 | ₐ |
| Nepali | 6000 | 0 |  |
| Serbian | 1000 | 0 |  |
| Venetian | 0 | 1 | Ẑ |
| Hindi | 6000 | 0 |  |
| Punjabi | 5992 | 0 |  |
| Eastern Yiddish | 0 | 8 | ֱֲֵֶֹֻׁׂ |
| Ligurian | 0 | 4 | ȇḗṉṻ |
| Icelandic | 3000 | 0 |  |
| Afrikaans | 0 | 1 | ȇ |
| Sinhala | 15000 | 0 |  |
| Urdu | 3000 | 1 | ؔ |
| Armenian | 5000 | 0 |  |
| Simplified Chinese | 0 | 3 | 噁奀岆 |
| Traditional Chinese | 0 | 98 | 凈噁婭屄撣淩狥狦猧猭獀獃獽玂琌琜琩瑈瑿璊 |
| Standard Arabic | 0 | 2 | ۖۚ |
| Egyptian Arabic | 0 | 1 | ۏ |
| Hebrew | 0 | 11 | ֱֲֳֵֶֹֺֻֽׁׂ |
| Tamil | 4000 | 0 |  |
| Malayalam | 1000 | 0 |  |
| North Azerbaijani | 1000 | 0 |  |
| Bashkir | 0 | 2 | ӦӰ |
| Tatar | 0 | 2 | ӥᴫ |
| Lao | 2000 | 0 |  |
| Japanese | 0 | 3 | ゙緖脋 |
| Korean | 0 | 48 | ᄙㅕㅛ걈굥귬꼮넫녬뉸늗닏댠돴뒥듁듌똣랃럅 |

Tier 3 summed over languages: 66,198
[tier 3] 1,695 s since the start


## 7. Vocabulary size per language

The number of tokens the output head computes for each language (S12.4): the base block, the script blocks of the scripts the
language uses (Qwen3-only tokens), and the language's tiers 1, 2 and 3. A language's scripts are read from its standard
characters.

In [16]:
rows = []
for language in QWEN3_LANGUAGES:
    own_scripts = sorted(language_scripts(language))
    script_tokens = sum(len(qwen_blocks.get(script, [])) for script in own_scripts)
    layers = [len(tier1[language.name]), len(tier2[language.name]), len(tier3[language.name])]
    rows.append([language.name, "+".join(own_scripts), script_tokens, *layers, base_size + script_tokens + sum(layers)])
table(rows, ["Language", "Scripts", "Script block tokens", "Tier 1", "Tier 2", "Tier 3", "Head size (with base block)"])
# Tier 2 forms are whole words and enter with their word-start mark.
new_strings = {"▁" + w for v in tier2.values() for w in v} | {w for v in tier3.values() for w in v}
new_strings -= vocabulary
script_total = sum(len(tokens) for name, tokens in qwen_blocks.items() if name != "base")
# These parts overlap (a tier 1 character can already be a Qwen3 token); section 9 counts the built tokenizer.
print(f"Base block {base_size:,}, script blocks {script_total:,}, tier 1 {len(all_tier1):,}, tiers 2 and 3 {len(new_strings):,}")
json.dump(rows, open(OUT / "vocab_size_by_language.json", "w"), ensure_ascii=False, indent=1)

| Language | Scripts | Script block tokens | Tier 1 | Tier 2 | Tier 3 | Head size (with base block) |
|---|---|---|---|---|---|---|
| English | latin | 68839 | 10 | 0 | 0 | 109152 |
| French | latin | 68839 | 23 | 0 | 0 | 109165 |
| Portuguese | latin | 68839 | 19 | 0 | 0 | 109161 |
| German | latin | 68839 | 11 | 0 | 0 | 109153 |
| Romanian | latin | 68839 | 16 | 0 | 0 | 109158 |
| Swedish | latin | 68839 | 10 | 0 | 0 | 109152 |
| Danish | latin | 68839 | 11 | 0 | 0 | 109153 |
| Bulgarian | cyrillic | 3949 | 19 | 0 | 3000 | 47271 |
| Russian | cyrillic | 3949 | 23 | 0 | 0 | 44275 |
| Czech | latin | 68839 | 28 | 750 | 0 | 109920 |
| Greek | greek | 193 | 53 | 0 | 5016 | 45565 |
| Ukrainian | cyrillic | 3949 | 25 | 0 | 1000 | 45277 |
| Spanish | latin | 68839 | 15 | 0 | 3 | 109160 |
| Dutch | latin | 68839 | 19 | 0 | 0 | 109161 |
| Slovak | latin | 68839 | 31 | 250 | 0 | 109423 |
| Croatian | latin | 68839 | 17 | 0 | 0 | 109159 |
| Polish | latin | 68839 | 21 | 0 | 0 | 109163 |
| Lithuanian | latin | 68839 | 25 | 0 | 0 | 109167 |
| Norwegian Bokmal | latin | 68839 | 14 | 0 | 0 | 109156 |
| Norwegian Nynorsk | latin | 68839 | 14 | 0 | 1 | 109157 |
| Persian | arabic | 3992 | 32 | 0 | 3000 | 47327 |
| Slovenian | latin | 68839 | 12 | 250 | 0 | 109404 |
| Gujarati | gujarati | 16 | 149 | 250 | 0 | 40718 |
| Latvian | latin | 68839 | 28 | 0 | 1000 | 110170 |
| Italian | latin | 68839 | 16 | 0 | 1 | 109159 |
| Occitan | latin | 68839 | 17 | 0 | 0 | 109159 |
| Nepali | devanagari | 63 | 154 | 0 | 6000 | 46520 |
| Marathi | devanagari | 63 | 156 | 0 | 0 | 40522 |
| Belarusian | cyrillic | 3949 | 25 | 0 | 0 | 44277 |
| Serbian | cyrillic | 3949 | 22 | 0 | 1000 | 45274 |
| Luxembourgish | latin | 68839 | 12 | 0 | 0 | 109154 |
| Venetian | latin | 68839 | 18 | 0 | 1 | 109161 |
| Assamese | bengali | 44 | 119 | 0 | 0 | 40466 |
| Welsh | latin | 68839 | 44 | 6000 | 0 | 115186 |
| Silesian | latin | 68839 | 23 | 0 | 0 | 109165 |
| Asturian | latin | 68839 | 20 | 0 | 0 | 109162 |
| Chhattisgarhi | devanagari | 63 | 164 | 0 | 0 | 40530 |
| Awadhi | devanagari | 63 | 164 | 0 | 0 | 40530 |
| Maithili | devanagari | 63 | 168 | 0 | 0 | 40534 |
| Bhojpuri | devanagari | 63 | 164 | 0 | 0 | 40530 |
| Sindhi | arabic | 3992 | 51 | 250 | 0 | 44596 |
| Irish | latin | 68839 | 13 | 0 | 0 | 109155 |
| Faroese | latin | 68839 | 17 | 0 | 0 | 109159 |
| Hindi | devanagari | 63 | 164 | 0 | 6000 | 46530 |
| Punjabi | gurmukhi | 10 | 142 | 750 | 5992 | 47197 |
| Bengali | bengali | 44 | 119 | 0 | 0 | 40466 |
| Oriya | oriya | 7 | 159 | 250 | 0 | 40719 |
| Tajik | cyrillic | 3949 | 32 | 0 | 0 | 44284 |
| Eastern Yiddish | hebrew | 3164 | 29 | 0 | 8 | 43504 |
| Lombard | latin | 68839 | 10 | 0 | 0 | 109152 |
| Ligurian | latin | 68839 | 27 | 0 | 4 | 109173 |
| Sicilian | latin | 68839 | 21 | 0 | 0 | 109163 |
| Friulian | latin | 68839 | 19 | 0 | 0 | 109161 |
| Sardinian | latin | 68839 | 16 | 0 | 0 | 109158 |
| Galician | latin | 68839 | 17 | 0 | 0 | 109159 |
| Catalan | latin | 68839 | 17 | 0 | 0 | 109159 |
| Icelandic | latin | 68839 | 18 | 0 | 3000 | 112160 |
| Tosk Albanian | latin | 68839 | 16 | 0 | 0 | 109158 |
| Limburgish | latin | 68839 | 19 | 0 | 0 | 109161 |
| Dari | arabic | 3992 | 32 | 0 | 0 | 44327 |
| Afrikaans | latin | 68839 | 17 | 0 | 1 | 109160 |
| Macedonian | cyrillic | 3949 | 24 | 500 | 0 | 44776 |
| Sinhala | sinhala | 26 | 145 | 0 | 15000 | 55474 |
| Urdu | arabic | 3992 | 42 | 0 | 3001 | 47338 |
| Magahi | devanagari | 63 | 164 | 0 | 0 | 40530 |
| Bosnian | latin | 68839 | 17 | 0 | 0 | 109159 |
| Armenian | armenian | 77 | 94 | 250 | 5000 | 45724 |
| Simplified Chinese | han | 25462 | 6850 | 0 | 3 | 72618 |
| Traditional Chinese | han | 25462 | 6801 | 0 | 98 | 72664 |
| Cantonese | han | 25462 | 6801 | 0 | 0 | 72566 |
| Burmese | myanmar | 37 | 218 | 4500 | 0 | 45058 |
| Standard Arabic | arabic | 3992 | 39 | 0 | 2 | 44336 |
| Najdi Arabic | arabic | 3992 | 29 | 0 | 0 | 44324 |
| Levantine Arabic | arabic | 3992 | 19 | 0 | 0 | 44314 |
| Egyptian Arabic | arabic | 3992 | 29 | 0 | 1 | 44325 |
| Moroccan Arabic | arabic | 3992 | 39 | 0 | 0 | 44334 |
| Mesopotamian Arabic | arabic | 3992 | 29 | 0 | 0 | 44324 |
| Taizzi-Adeni Arabic | arabic | 3992 | 29 | 0 | 0 | 44324 |
| Tunisian Arabic | arabic | 3992 | 39 | 0 | 0 | 44334 |
| Hebrew | hebrew | 3164 | 15 | 0 | 11 | 43493 |
| Maltese | latin | 68839 | 24 | 0 | 0 | 109166 |
| Indonesian | latin | 68839 | 10 | 0 | 0 | 109152 |
| Malay | latin | 68839 | 10 | 0 | 0 | 109152 |
| Tagalog | latin | 68839 | 12 | 0 | 0 | 109154 |
| Cebuano | latin | 68839 | 10 | 0 | 0 | 109152 |
| Javanese | latin | 68839 | 37 | 0 | 0 | 109179 |
| Sundanese | latin | 68839 | 10 | 0 | 0 | 109152 |
| Minangkabau | latin | 68839 | 10 | 0 | 0 | 109152 |
| Balinese | latin | 68839 | 10 | 0 | 0 | 109152 |
| Banjar | latin | 68839 | 10 | 0 | 0 | 109152 |
| Pangasinan | latin | 68839 | 12 | 0 | 0 | 109154 |
| Iloko | latin | 68839 | 12 | 0 | 0 | 109154 |
| Waray | latin | 68839 | 12 | 0 | 0 | 109154 |
| Tamil | tamil | 28 | 102 | 0 | 4000 | 44433 |
| Telugu | telugu | 14 | 164 | 0 | 0 | 40481 |
| Kannada | kannada | 25 | 144 | 0 | 0 | 40472 |
| Malayalam | malayalam | 32 | 175 | 0 | 1000 | 41510 |
| Turkish | latin | 68839 | 13 | 0 | 0 | 109155 |
| North Azerbaijani | latin | 68839 | 15 | 0 | 1000 | 110157 |
| Northern Uzbek | latin | 68839 | 15 | 0 | 0 | 109157 |
| Kazakh | cyrillic | 3949 | 40 | 0 | 0 | 44292 |
| Bashkir | cyrillic | 3949 | 42 | 0 | 2 | 44296 |
| Tatar | cyrillic | 3949 | 36 | 0 | 2 | 44290 |
| Thai | thai | 2570 | 84 | 0 | 0 | 42957 |
| Lao | lao | 33 | 127 | 0 | 2000 | 42463 |
| Finnish | latin | 68839 | 11 | 250 | 0 | 109403 |
| Estonian | latin | 68839 | 13 | 0 | 0 | 109155 |
| Hungarian | latin | 68839 | 38 | 0 | 0 | 109180 |
| Vietnamese | latin | 68839 | 120 | 0 | 0 | 109262 |
| Khmer | khmer | 36 | 155 | 2250 | 0 | 42744 |
| Japanese | han+kana | 27529 | 8817 | 0 | 3 | 76652 |
| Korean | hangul | 3564 | 2432 | 3750 | 48 | 50097 |
| Georgian | georgian | 37 | 215 | 0 | 0 | 40555 |
| Basque | latin | 68839 | 12 | 0 | 0 | 109154 |
| Haitian | latin | 68839 | 12 | 0 | 0 | 109154 |
| Papiamento | latin | 68839 | 16 | 0 | 0 | 109158 |
| Kabuverdianu | latin | 68839 | 13 | 0 | 0 | 109155 |
| Tok Pisin | latin | 68839 | 10 | 0 | 0 | 109152 |
| Swahili | latin | 68839 | 10 | 0 | 0 | 109152 |

Base block 40,303, script blocks 115,535, tier 1 18,372, tiers 2 and 3 84,396


## 8. Final check

The final vocabulary (Qwen3 and Helium tokens, tiers 1, 2 and 3) gets the Unigram EM of section 3 on the word counts of the
text tier 2 was chosen from (the training parts and the YODAS2 subtitles of every language, words seen at least twice), as the
model will cut text. With only the 30,000-line EM sample, tier 2 forms that sample never holds get the lowest score and are
not used as one token. Each language's evaluation cut rate is then measured with that model
and compared with its target (S2.2).

Qwen3 tokens of two or more characters that this EM never uses are then dropped (S12.1): words merged from written text, such
as Chinese `劳动力` or English words, that speech does not use. The same rule holds for every script. Kept whatever their use:
Helium tokens (ids and head rows), single characters, tiers 1 to 3, code pieces for tool calls (tokens with ASCII punctuation,
camel case inside, or `<...>` markers), emoji and single-character symbols. Two more kinds are dropped when unused: runs of
symbols with no letter or digit (`)$$`, `--,`), except runs made only of emoji, and the word-start forms of single Han and
kana characters, since `▁` comes there only at a phrase start. The EM then runs again on the smaller vocabulary and the cut rates are
measured with it.

In [17]:
final_pieces = sorted(vocabulary | new_strings)
final_counts = collections.Counter()
for counts in unit_counts.values():
    final_counts.update({word: count for word, count in counts.items() if count >= 2})
print(f"Unigram EM on the final vocabulary ({len(final_pieces):,} tokens), {len(final_counts):,} distinct words, "
      f"{sum(final_counts.values()):,} in all")
unigram_final = fit_unigram(final_pieces, final_counts)


def emoji(ch):
    return 0x1F000 <= ord(ch) <= 0x1FAFF or 0x2600 <= ord(ch) <= 0x27BF or ch in "\u200d\ufe0f"


def droppable(piece):
    core = piece.lstrip("▁")
    if piece in helium_pieces:
        return False
    # Han and kana have no spaces, so `▁` comes only at a phrase start and a rare character's word-start form goes unused.
    if piece.startswith("▁") and len(core) == 1 and script_of(core) in ("han", "kana"):
        return True
    if piece in kept_pieces or len(core) < 2:
        return False
    if any(unicodedata.category(ch)[0] == "S" or ch in "\u200d\ufe0f" for ch in core):
        # A run of symbols with no letter or digit is markdown or code formatting, unless it is all emoji.
        return not any(ch.isalnum() for ch in core) and not all(emoji(ch) for ch in core)
    code = any(ch.isascii() and unicodedata.category(ch)[0] == "P" for ch in core) or any(ch.isupper() for ch in core[1:])
    return not code and not (core.startswith("<") and core.endswith(">"))


helium_pieces = {as_piece(surface) for surface in helium_surface}
kept_pieces = helium_pieces | set(unigram_pieces(tier1_all)) - set(base_pieces) | new_strings
lowest_score = min(unigram_final.get_score(unigram_final.piece_to_id(piece)) for piece in final_pieces)
dropped = {piece for piece in final_pieces
           if unigram_final.get_score(unigram_final.piece_to_id(piece)) == lowest_score and droppable(piece)}
by_script = collections.Counter(block_of(piece.replace("▁", " ")) for piece in dropped)
print(f"Dropped {len(dropped):,} unused tokens: " + ", ".join(f"{name} {count:,}" for name, count in by_script.most_common(8)))
final_pieces = [piece for piece in final_pieces if piece not in dropped]
unigram_final = fit_unigram(final_pieces, final_counts)
(OUT / "unigram_final.model").write_bytes(unigram_final.serialized_model_proto())

SHORT_SPOKEN_LINES = 5000
rows, met = [], 0
heads = {row[0]: row[-1] for row in json.load(open(OUT / "vocab_size_by_language.json"))}
for language in QWEN3_LANGUAGES:
    if language.name not in tier2_state:
        continue
    layouts, target, _ = tier2_state[language.name]
    words = sorted({word for layout_words, *_ in layouts for word in layout_words})
    counts = dict(zip(words, map(len, unigram_final.encode(words, num_threads=THREADS))))
    rate = cut_rate(layouts, counter=counts.__getitem__)
    met += rate < target
    head = heads[language.name] - sum(by_script[script] for script in language_scripts(language))
    lines = len(splits[language.name][0])
    # A language with little spoken text is left above its target rather than given written text (S2.3).
    note = f"spoken data short ({lines:,} training lines)" if rate >= target and lines < SHORT_SPOKEN_LINES else ""
    rows.append([language.name, len(tier1[language.name]), len(tier2[language.name]), len(tier3[language.name]),
                 head, f"{rate:.2%}", f"{target:.2%}", "yes" if rate < target else "no", note])
table(rows, ["Language", "Tier 1", "Tier 2", "Tier 3", "Head size", "Final cut rate", "Target", "Below target", "Note"])
print(f"Languages below their target: {met}/{len(rows)}")
json.dump(rows, open(OUT / "final_check.json", "w"), ensure_ascii=False, indent=1)
elapsed("final check")

Unigram EM on the final vocabulary (254,366 tokens), 3,129,140 distinct words, 104,828,151 in all


  EM 1: tokens used 183,469/254,366


  EM 2: tokens used 180,481/254,366


  EM 3: tokens used 179,503/254,366


  EM 4: tokens used 178,928/254,366


Dropped 30,685 unused tokens: han 17,352, latin 10,674, base 1,235, thai 583, kana 568, cyrillic 122, arabic 83, hebrew 59


... (2 outputs left out)

  EM 3: tokens used 179,073/223,681


  EM 4: tokens used 178,914/223,681


| Language | Tier 1 | Tier 2 | Tier 3 | Head size | Final cut rate | Target | Below target | Note |
|---|---|---|---|---|---|---|---|---|
| English | 10 | 0 | 0 | 98478 | 0.18% | 0.28% | yes |  |
| French | 23 | 0 | 0 | 98491 | 0.54% | 1.00% | yes |  |
| Portuguese | 19 | 0 | 0 | 98487 | 0.29% | 1.00% | yes |  |
| German | 11 | 0 | 0 | 98479 | 0.07% | 1.00% | yes |  |
| Romanian | 16 | 0 | 0 | 98484 | 0.57% | 1.00% | yes |  |
| Swedish | 10 | 0 | 0 | 98478 | 0.55% | 1.00% | yes |  |
| Danish | 11 | 0 | 0 | 98479 | 0.83% | 1.00% | yes |  |
| Bulgarian | 19 | 0 | 3000 | 47149 | 0.87% | 1.00% | yes |  |
| Russian | 23 | 0 | 0 | 44153 | 0.32% | 1.00% | yes |  |
| Czech | 28 | 750 | 0 | 99246 | 0.72% | 1.00% | yes |  |
| Greek | 53 | 0 | 5016 | 45565 | 0.88% | 1.00% | yes |  |
| Ukrainian | 25 | 0 | 1000 | 45155 | 0.80% | 1.00% | yes |  |
| Spanish | 15 | 0 | 3 | 98486 | 0.23% | 1.00% | yes |  |
| Dutch | 19 | 0 | 0 | 98487 | 0.29% | 1.00% | yes |  |
| Slovak | 31 | 250 | 0 | 98749 | 0.66% | 1.00% | yes |  |
| Croatian | 17 | 0 | 0 | 98485 | 0.31% | 1.00% | yes |  |
| Polish | 21 | 0 | 0 | 98489 | 0.44% | 1.00% | yes |  |
| Lithuanian | 25 | 0 | 0 | 98493 | 0.44% | 1.00% | yes |  |
| Norwegian Bokmal | 14 | 0 | 0 | 98482 | 0.43% | 1.00% | yes |  |
| Norwegian Nynorsk | 14 | 0 | 1 | 98483 | 0.01% | 1.00% | yes |  |
| Persian | 32 | 0 | 3000 | 47244 | 0.59% | 1.00% | yes |  |
| Slovenian | 12 | 250 | 0 | 98730 | 0.50% | 1.00% | yes |  |
| Gujarati | 149 | 250 | 0 | 40718 | 0.71% | 1.00% | yes |  |
| Latvian | 28 | 0 | 1000 | 99496 | 0.66% | 1.00% | yes |  |
| Italian | 16 | 0 | 1 | 98485 | 0.31% | 1.00% | yes |  |
| Occitan | 17 | 0 | 0 | 98485 | 0.61% | 1.00% | yes |  |
| Nepali | 154 | 0 | 6000 | 46520 | 0.53% | 1.00% | yes |  |
| Marathi | 156 | 0 | 0 | 40522 | 0.07% | 1.00% | yes |  |
| Belarusian | 25 | 0 | 0 | 44155 | 0.10% | 1.00% | yes |  |
| Serbian | 22 | 0 | 1000 | 45152 | 0.82% | 1.00% | yes |  |
| Luxembourgish | 12 | 0 | 0 | 98480 | 0.29% | 1.00% | yes |  |
| Venetian | 18 | 0 | 1 | 98487 | 0.00% | 1.00% | yes |  |
| Assamese | 119 | 0 | 0 | 40466 | 0.32% | 1.00% | yes |  |
| Welsh | 44 | 6000 | 0 | 104512 | 0.93% | 1.00% | yes |  |
| Silesian | 23 | 0 | 0 | 98491 | 0.00% | 1.00% | yes |  |
| Asturian | 20 | 0 | 0 | 98488 | 0.17% | 1.00% | yes |  |
| Maithili | 168 | 0 | 0 | 40534 | 0.00% | 1.00% | yes |  |
| Bhojpuri | 164 | 0 | 0 | 40530 | 0.02% | 1.00% | yes |  |
| Sindhi | 51 | 250 | 0 | 44513 | 0.21% | 1.00% | yes |  |
| Irish | 13 | 0 | 0 | 98481 | 0.00% | 1.00% | yes |  |
| Faroese | 17 | 0 | 0 | 98485 | 0.00% | 1.00% | yes |  |
| Hindi | 164 | 0 | 6000 | 46530 | 0.50% | 1.00% | yes |  |
| Punjabi | 142 | 750 | 5992 | 47197 | 1.97% | 1.00% | no | spoken data short (2,198 training lines) |
| Bengali | 119 | 0 | 0 | 40466 | 0.08% | 1.00% | yes |  |
| Oriya | 159 | 250 | 0 | 40719 | 0.79% | 1.00% | yes |  |
| Tajik | 32 | 0 | 0 | 44162 | 0.08% | 1.00% | yes |  |
| Eastern Yiddish | 29 | 0 | 8 | 43445 | 0.00% | 1.00% | yes |  |
| Lombard | 10 | 0 | 0 | 98478 | 0.00% | 1.00% | yes |  |
| Ligurian | 27 | 0 | 4 | 98499 | 0.02% | 1.00% | yes |  |
| Sicilian | 21 | 0 | 0 | 98489 | 0.00% | 1.00% | yes |  |
| Friulian | 19 | 0 | 0 | 98487 | 0.01% | 1.00% | yes |  |
| Sardinian | 16 | 0 | 0 | 98484 | 0.00% | 1.00% | yes |  |
| Galician | 17 | 0 | 0 | 98485 | 0.46% | 1.00% | yes |  |
| Catalan | 17 | 0 | 0 | 98485 | 0.81% | 1.00% | yes |  |
| Icelandic | 18 | 0 | 3000 | 101486 | 0.77% | 1.00% | yes |  |
| Tosk Albanian | 16 | 0 | 0 | 98484 | 0.00% | 1.00% | yes |  |
| Limburgish | 19 | 0 | 0 | 98487 | 0.00% | 1.00% | yes |  |
| Afrikaans | 17 | 0 | 1 | 98486 | 0.16% | 1.00% | yes |  |
| Macedonian | 24 | 500 | 0 | 44654 | 0.43% | 1.00% | yes |  |
| Sinhala | 145 | 0 | 15000 | 55474 | 0.95% | 1.00% | yes |  |
| Urdu | 42 | 0 | 3001 | 47255 | 0.75% | 1.00% | yes |  |
| Bosnian | 17 | 0 | 0 | 98485 | 0.28% | 1.00% | yes |  |
| Armenian | 94 | 250 | 5000 | 45724 | 0.08% | 1.00% | yes |  |
| Simplified Chinese | 6850 | 0 | 3 | 55266 | 0.00% | 0.20% | yes |  |
| Traditional Chinese | 6801 | 0 | 98 | 55312 | 0.03% | 0.20% | yes |  |
| Cantonese | 6801 | 0 | 0 | 55214 | 0.01% | 1.00% | yes |  |
| Burmese | 218 | 4500 | 0 | 45058 | 0.68% | 1.00% | yes |  |
| Standard Arabic | 39 | 0 | 2 | 44253 | 0.45% | 1.00% | yes |  |
| Egyptian Arabic | 29 | 0 | 1 | 44242 | 0.01% | 1.00% | yes |  |
| Moroccan Arabic | 39 | 0 | 0 | 44251 | 0.01% | 1.00% | yes |  |
| Hebrew | 15 | 0 | 11 | 43434 | 0.34% | 1.00% | yes |  |
| Maltese | 24 | 0 | 0 | 98492 | 0.01% | 1.00% | yes |  |
| Indonesian | 10 | 0 | 0 | 98478 | 0.13% | 1.00% | yes |  |
| Malay | 10 | 0 | 0 | 98478 | 0.30% | 1.00% | yes |  |
| Tagalog | 12 | 0 | 0 | 98480 | 0.01% | 1.00% | yes |  |
| Cebuano | 10 | 0 | 0 | 98478 | 0.01% | 1.00% | yes |  |
| Javanese | 37 | 0 | 0 | 98505 | 0.01% | 1.00% | yes |  |
| Sundanese | 10 | 0 | 0 | 98478 | 0.02% | 1.00% | yes |  |
| Minangkabau | 10 | 0 | 0 | 98478 | 0.00% | 1.00% | yes |  |
| Balinese | 10 | 0 | 0 | 98478 | 0.08% | 1.00% | yes |  |
| Banjar | 10 | 0 | 0 | 98478 | 0.01% | 1.00% | yes |  |
| Pangasinan | 12 | 0 | 0 | 98480 | 0.00% | 1.00% | yes |  |
| Iloko | 12 | 0 | 0 | 98480 | 0.01% | 1.00% | yes |  |
| Waray | 12 | 0 | 0 | 98480 | 0.00% | 1.00% | yes |  |
| Tamil | 102 | 0 | 4000 | 44433 | 0.79% | 1.00% | yes |  |
| Telugu | 164 | 0 | 0 | 40481 | 0.30% | 1.00% | yes |  |
| Kannada | 144 | 0 | 0 | 40472 | 0.27% | 1.00% | yes |  |
| Malayalam | 175 | 0 | 1000 | 41510 | 0.23% | 1.00% | yes |  |
| Turkish | 13 | 0 | 0 | 98481 | 0.14% | 1.00% | yes |  |
| North Azerbaijani | 15 | 0 | 1000 | 99483 | 0.79% | 1.00% | yes |  |
| Northern Uzbek | 15 | 0 | 0 | 98483 | 0.46% | 1.00% | yes |  |
| Kazakh | 40 | 0 | 0 | 44170 | 0.00% | 1.00% | yes |  |
| Bashkir | 42 | 0 | 2 | 44174 | 0.01% | 1.00% | yes |  |
| Tatar | 36 | 0 | 2 | 44168 | 0.04% | 1.00% | yes |  |
| Thai | 84 | 0 | 0 | 42374 | 0.45% | 1.00% | yes |  |
| Lao | 127 | 0 | 2000 | 42463 | 0.73% | 1.00% | yes |  |
| Finnish | 11 | 250 | 0 | 98729 | 0.32% | 1.00% | yes |  |
| Estonian | 13 | 0 | 0 | 98481 | 0.78% | 1.00% | yes |  |
| Hungarian | 38 | 0 | 0 | 98506 | 0.36% | 1.00% | yes |  |
| Vietnamese | 120 | 0 | 0 | 98588 | 0.81% | 1.00% | yes |  |
| Khmer | 155 | 2250 | 0 | 42744 | 0.61% | 1.00% | yes |  |
| Japanese | 8817 | 0 | 3 | 58732 | 0.00% | 0.20% | yes |  |
| Korean | 2432 | 3750 | 48 | 50093 | 0.23% | 0.23% | yes |  |
| Georgian | 215 | 0 | 0 | 40555 | 0.45% | 1.00% | yes |  |
| Basque | 12 | 0 | 0 | 98480 | 0.34% | 1.00% | yes |  |
| Haitian | 12 | 0 | 0 | 98480 | 0.14% | 1.00% | yes |  |
| Papiamento | 16 | 0 | 0 | 98484 | 0.00% | 1.00% | yes |  |
| Kabuverdianu | 13 | 0 | 0 | 98481 | 0.00% | 1.00% | yes |  |
| Tok Pisin | 10 | 0 | 0 | 98478 | 0.00% | 1.00% | yes |  |
| Swahili | 10 | 0 | 0 | 98478 | 0.00% | 1.00% | yes |  |

Languages below their target: 109/110
[final check] 1,814 s since the start
